# Original EyeModes Joint — dataset batch, one prompt
**ฉบับแก้ v3 — ใช้ตัวอย่างล่าสุดโดยตรง: HairFastGAN_Diffusion_Two_Modes (7).ipynb จริง ๆ**

คัดลอก cell ฟังก์ชัน diffusion / Joint / eye protection / prompts จากต้นฉบับทั้ง cell แบบไม่แก้เนื้อหา พร้อมบันทึก SHA256 ตรวจสอบได้

ใช้ `all_prompts_joint_controlled` และเลือกเฉพาะ `prompt6_combined` หนึ่ง prompt, seed3407, strength0.5, steps20, guidance7.5, face blend0.35 ตามค่าเดิม โหมดนี้เก็บพิกเซลตาจากภาพก่อน diffusion ไม่ใช่โหมด eyerefine

**ไม่มี RGB lock, color correction, sharpen พื้นหลัง, หรือ fallback คืนภาพเดิม** ผมและหน้าสามารถเปลี่ยนตาม Joint mask ได้เหมือนเดิม สีถูกขอให้รักษาใน prompt ไม่ใช่การบังคับให้พิกเซลสีเท่ากันทั้งหมด

ภาพเข้า: ใช้ HairFastGAN Rotate ON/OFF และ Barbershop ก่อน diffusion ที่สร้างไว้แล้ว น้ำหนัก/epoch ยึดตามรายงานต้นทาง ไม่ฝึกใหม่ ไม่เปลี่ยน checkpoint และไม่สร้าง rotation images ซ้ำ

เปิด Colab GPU → Run all → เชื่อม Drive หากตรวจตาได้ครบจะทำต่อจนจบ หากตรวจตาไม่ได้ จะลองย่อภาพ/เพิ่มขอบด้วยตัวตรวจเดิมและแปลงพิกัดกลับอัตโนมัติ ภาพที่พร้อมทำต่อได้ หากยังค้างจึงใช้ช่องตรวจตำแหน่งตา แล้วรันเซลล์สร้างภาพต่อ **ไม่เดาตำแหน่ง ไม่ข้ามคู่ข้อมูล ไม่แอบแทนด้วยภาพก่อน diffusion**

เพิ่ม eye_debug.zip: ภาพที่ตัวตรวจได้รับจริง จำนวนใบหน้าที่พบ และรุ่นแพ็กเกจ หากยังมีภาพค้างจะดาวน์โหลดไฟล์นี้ให้ส่งตรวจต่อ โดยไม่อ้างว่าพบสาเหตุแล้ว

ตั้ง SOURCE_RUN เป็นรอบเดิมของคุณแล้ว ผลใหม่อยู่ `original_joint_...` หาก runtime หลุดใส่ RESUME_OUTPUT เพื่อรันต่อ

ฟังก์ชัน diffusion และ prompt ตรงต้นฉบับ; เพิ่มเฉพาะการลองตรวจตาซ้ำสำหรับภาพที่เดิมตรวจไม่ผ่าน แต่ไม่รับประกันผลเหมือนทุกพิกเซลข้าม GPU/package/model revision และไม่รับประกันว่าภาพทุกภาพจะดีขึ้น ต้องตรวจผลจริง


In [ ]:
SOURCE_NOTEBOOK_SHA256='d7f0ffcfad5863e2caf7353d7e26e9c619bd3a5a1cd26c9b6d90517c63e37a85'
ORIGINAL_HELPER_SHA256='b6b147aeb041dfb379cc96d4c98340080c1c88a80ad7f83d009323c10640e839'
from google.colab import drive,files
from pathlib import Path
import os,sys,json,hashlib,shutil,subprocess,time,uuid,gc,copy
import numpy as np,pandas as pd
from PIL import Image,ImageOps,ImageFilter,ImageDraw

drive.mount('/content/drive')
SOURCE_RUN='/content/drive/MyDrive/Rotation_Diffusion_EyeMask_Study/strict_hair_bg_20261009_071358_e864f4'
SOURCE_REPORT='' # ปกติหาได้จาก provenance ของรอบเดิม; กรอกเมื่อย้ายโฟลเดอร์
RESUME_OUTPUT='' # ว่าง=รอบใหม่; ใส่โฟลเดอร์ original_joint_... เพื่อรันต่อหลัง runtime หลุด
ROOT=Path('/content/drive/MyDrive/Rotation_Diffusion_EyeMask_Study')

def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for part in iter(lambda:f.read(8*1024**2),b''):h.update(part)
    return h.hexdigest()
def need(path):
    if not path.is_file():raise FileNotFoundError('ต้องใช้ไฟล์เดิม: '+str(path))
    return path
source=Path(SOURCE_RUN)
original_records=json.loads(need(source/'base_records.json').read_text())
ancestor=source;visited=set();old_hashes={}
while str(ancestor) not in visited:
    visited.add(str(ancestor))
    if (ancestor/'input_hashes.json').is_file():
        old_hashes=json.loads((ancestor/'input_hashes.json').read_text());break
    p=ancestor/'source_run.json'
    if not p.is_file():break
    ancestor=Path(json.loads(p.read_text())['previous_output'])
if SOURCE_REPORT:REPORT_ROOT=Path(SOURCE_REPORT)
else:
    candidates=set()
    for name in old_hashes:
        p=Path(name)
        if len(p.parents)>=4 and p.parent.parent.name=='images':
            root=p.parents[3]
            if (root/'common_protocol.json').is_file():candidates.add(root)
    assert len(candidates)==1,'หา common report ต้นทางไม่ชัดเจน: ใส่ SOURCE_REPORT ด้านบน'
    REPORT_ROOT=candidates.pop()
SOURCE_PROTOCOL=json.loads(need(REPORT_ROOT/'common_protocol.json').read_text())
SOURCE_LOCK=json.loads(need(REPORT_ROOT/'checkpoint_lock.json').read_text())
SPLITS=SOURCE_PROTOCOL['splits'];TRIALS=['Baseline','A','B','C','D']
BRANCHES=['HFG_rotate_on','HFG_rotate_off','Barbershop']
expected={(s,t,k,b) for s,keys in SPLITS.items() for t in TRIALS for k in keys for b in BRANCHES}
assert len(original_records)==len(expected)
assert {(r['split'],r['trial'],r['case'],r['branch']) for r in original_records}==expected
for r in original_records:assert sha(need(Path(r['path'])))==r['sha256']
INPUT_HASHES={}
for split,keys in SPLITS.items():
    for key in keys:
        for field in ['face','ref','shape','target','protected','eval_core','eval_refmask']:
            p=need(REPORT_ROOT/split/'images'/key/(field+'.png'))
            INPUT_HASHES[str(p)]=sha(p)
            if str(p) in old_hashes:assert INPUT_HASHES[str(p)]==old_hashes[str(p)]
# Keep original defaults; these are checked against the copied source cell later.
MODE='all_prompts_joint_controlled'
PROMPT_TAG='prompt6_combined'
SEED=3407
CONFIG=dict(version='original_joint_batch_v3_latest_example_audit',source_run=str(source),source_report=str(REPORT_ROOT),
    source_base_records_sha256=sha(source/'base_records.json'),mode=MODE,prompt_tag=PROMPT_TAG,
    seed=SEED,strength=.5,steps=20,guidance=7.5,face_blend=.35,
    helper_sha256=ORIGINAL_HELPER_SHA256,source_notebook_sha256=SOURCE_NOTEBOOK_SHA256,
    failure_policy='original detector then fixed resize/pad retries; process ready images; require full cohort for reports; no substituted outputs',
    method='original Joint and prompts; batch eye recovery with same detector, mapped to original coordinates; no RGB lock or sharpening')
if RESUME_OUTPUT:
    OUT=Path(RESUME_OUTPUT);assert json.loads(need(OUT/'run_config.json').read_text())==CONFIG
    BASE_RECORDS=json.loads(need(OUT/'base_records.json').read_text())
else:
    OUT=ROOT/('original_joint_'+time.strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6]);OUT.mkdir(parents=True)
    BASE_RECORDS=[]
    for r in original_records:
        folder=OUT/r['split']/r['trial']/r['case']/r['branch'];folder.mkdir(parents=True)
        dest=folder/'before.png';shutil.copy2(r['path'],dest)
        BASE_RECORDS.append(dict(r,path=str(dest),sha256=sha(dest)))
    (OUT/'base_records.json').write_text(json.dumps(BASE_RECORDS,indent=2))
    (OUT/'run_config.json').write_text(json.dumps(CONFIG,indent=2))
RESULTS_ROOT=OUT
NEED_EVALUATOR=not all(n in globals() for n in ['hair_fast','id_net','parser_net','shape_metrics','extra_metrics','quantile_color','read_mask','gpu'])
print('Output:',OUT,'| expected before:',len(BASE_RECORDS),'| expected after:',len(BASE_RECORDS))
print('Original Joint method: reuse pre-diffusion images. One prompt/seed. No training, RGB lock or sharpening.')


In [ ]:
%pip install -q opencv-python matplotlib scipy scikit-image scikit-learn pandas tqdm dlib face-alignment gdown addict wandb torchmetrics dill ninja transformers diffusers accelerate fastapi uvicorn python-multipart pyngrok nest-asyncio huggingface_hub
%pip install -q git+https://github.com/openai/CLIP.git
import torch
assert torch.cuda.is_available(), 'เลือก Runtime > Change runtime type > GPU ก่อนรัน'
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)
%pip install -q ipywidgets

In [ ]:
if NEED_EVALUATOR:
    from google.colab import drive
    from pathlib import Path
    import os, shutil, subprocess, sys, json, time
    from huggingface_hub import snapshot_download
    
    
    LOCAL_REPO = '/content/HairFastGAN_rotation_eyes'
    DRIVE_CACHE_DIR = Path('/content/drive/MyDrive/HairFastGAN_cache/pretrained_models')
    
    # ใส่ commit ที่บันทึกใน metadata.json เมื่อต้องการทำซ้ำภายหลัง
    HAIRFAST_REVISION = '49e98019c9243d1991c19d45764fc3340a498de2'
    if not Path(LOCAL_REPO, '.git').exists():
        subprocess.run(['git', 'clone', 'https://github.com/AIRI-Institute/HairFastGAN.git', LOCAL_REPO], check=True)
    if HAIRFAST_REVISION:
        subprocess.run(['git', '-C', LOCAL_REPO, 'checkout', HAIRFAST_REVISION], check=True)
    os.chdir(LOCAL_REPO)
    local_models = Path(LOCAL_REPO, 'pretrained_models')
    if not local_models.exists() or not any(local_models.iterdir()):
        if DRIVE_CACHE_DIR.exists() and any(DRIVE_CACHE_DIR.iterdir()):
            shutil.copytree(DRIVE_CACHE_DIR, local_models, dirs_exist_ok=True)
        else:
            snapshot = snapshot_download('AIRI-Institute/HairFastGAN', allow_patterns=['pretrained_models/**'])
            shutil.copytree(Path(snapshot, 'pretrained_models'), local_models, dirs_exist_ok=True)
    
    # ตรวจ cache สำคัญ รวมถึงกรณีได้ Git-LFS pointer แทน weights
    for relative in ['StyleGAN/ffhq.pt', 'encoder4editing/e4e_ffhq_encode.pt']:
        p = local_models / relative
        if not p.exists() or p.stat().st_size < 1000000:
            raise RuntimeError(f'โมเดลขาดหรือไม่สมบูรณ์: {p} โปรดตรวจ cache ก่อนดำเนินการ')
    Path('input').mkdir(exist_ok=True)
    Path('output').mkdir(exist_ok=True)
    REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
    print('Repo commit:', REPO_COMMIT)


In [ ]:
if NEED_EVALUATOR:
    # Compatibility backend: native torch operators also execute on CUDA tensors.
    # No PyTorch/CUDA version downgrade; no changes to weights or transfer parameters.
    import ast, hashlib, importlib, sys
    from pathlib import Path
    STYLEGAN_OPS_BACKEND = 'torch_native'
    
    def _native_stylegan_source(source, kind):
        marker = '# HAIRFAST_TORCH_NATIVE_V1'
        if marker in source:
            return source
        tree = ast.parse(source)
        extension = 'fused' if kind == 'fused_act' else 'upfirdn2d_op'
        entry = 'fused_leaky_relu' if kind == 'fused_act' else 'upfirdn2d'
        edits = []
        found_load = found_entry = False
        for node in tree.body:
            if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == extension for t in node.targets):
                if not isinstance(node.value, ast.Call) or not isinstance(node.value.func, ast.Name) or node.value.func.id != 'load':
                    raise RuntimeError('รูปแบบ extension เปลี่ยนไป: ' + kind)
                edits.append((node.lineno-1, node.end_lineno, extension + ' = None\n'))
                found_load = True
            if isinstance(node, ast.FunctionDef) and node.name == entry:
                edits.append((node.lineno-1, node.end_lineno, FUSED_NATIVE if kind == 'fused_act' else UP_NATIVE))
                found_entry = True
        if not found_load or not found_entry:
            raise RuntimeError('ไม่พบจุด patch ที่คาดไว้ใน ' + kind)
        lines = source.splitlines(True)
        for start,end,text in sorted(edits, reverse=True):
            lines[start:end] = [text]
        updated = marker + '\n' + ''.join(lines)
        if kind == 'upfirdn2d' and not any(isinstance(n, ast.FunctionDef) and n.name == 'upfirdn2d_native' for n in tree.body):
            updated += '\n\n' + UP_NATIVE_IMPL + '\n'
        compile(updated, kind + '.py', 'exec')
        return updated
    
    FUSED_NATIVE = 'def fused_leaky_relu(input, bias=None, negative_slope=0.2, scale=2 ** 0.5):\n    if bias is not None:\n        input = input + bias.reshape(1, -1, *([1] * (input.ndim - 2)))\n    return torch.nn.functional.leaky_relu(input, negative_slope=negative_slope) * scale\n'
    UP_NATIVE = 'def upfirdn2d(input, kernel, up=1, down=1, pad=(0, 0)):\n    return upfirdn2d_native(input, kernel, up, up, down, down,\n                           pad[0], pad[1], pad[0], pad[1])\n'
    UP_NATIVE_IMPL = 'def upfirdn2d_native(\n    input, kernel, up_x, up_y, down_x, down_y, pad_x0, pad_x1, pad_y0, pad_y1\n):\n    _, channel, in_h, in_w = input.shape\n    input = input.reshape(-1, in_h, in_w, 1)\n\n    _, in_h, in_w, minor = input.shape\n    kernel_h, kernel_w = kernel.shape\n\n    out = input.view(-1, in_h, 1, in_w, 1, minor)\n    out = F.pad(out, [0, 0, 0, up_x - 1, 0, 0, 0, up_y - 1])\n    out = out.view(-1, in_h * up_y, in_w * up_x, minor)\n\n    out = F.pad(\n        out, [0, 0, max(pad_x0, 0), max(pad_x1, 0), max(pad_y0, 0), max(pad_y1, 0)]\n    )\n    out = out[\n        :,\n        max(-pad_y0, 0) : out.shape[1] - max(-pad_y1, 0),\n        max(-pad_x0, 0) : out.shape[2] - max(-pad_x1, 0),\n        :,\n    ]\n\n    out = out.permute(0, 3, 1, 2)\n    out = out.reshape(\n        [-1, 1, in_h * up_y + pad_y0 + pad_y1, in_w * up_x + pad_x0 + pad_x1]\n    )\n    w = torch.flip(kernel, [0, 1]).view(1, 1, kernel_h, kernel_w)\n    out = F.conv2d(out, w)\n    out = out.reshape(\n        -1,\n        minor,\n        in_h * up_y + pad_y0 + pad_y1 - kernel_h + 1,\n        in_w * up_x + pad_x0 + pad_x1 - kernel_w + 1,\n    )\n    out = out.permute(0, 2, 3, 1)\n    out = out[:, ::down_y, ::down_x, :]\n\n    out_h = (in_h * up_y + pad_y0 + pad_y1 - kernel_h) // down_y + 1\n    out_w = (in_w * up_x + pad_x0 + pad_x1 - kernel_w) // down_x + 1\n\n    return out.view(-1, channel, out_h, out_w)'
    
    def install_native_stylegan_ops(repo):
        root = Path(repo).resolve()
        pairs = sorted((root / 'models').rglob('op/fused_act.py'))
        if not pairs:
            raise RuntimeError('ไม่พบ StyleGAN operator ใน ' + str(root))
        planned = []
        for fused_path in pairs:
            for path,kind in [(fused_path,'fused_act'), (fused_path.with_name('upfirdn2d.py'),'upfirdn2d')]:
                if not path.is_file():
                    raise RuntimeError('ขาด operator: ' + str(path))
                module_name = '.'.join(path.relative_to(root).with_suffix('').parts)
                if module_name in sys.modules:
                    raise RuntimeError('StyleGAN ถูก import แล้ว: Restart runtime แล้วรันตามลำดับใหม่')
                old = path.read_text()
                planned.append((path, old, _native_stylegan_source(old,kind)))
        # Validate every candidate before writing; preserve each original once.
        records = []
        for path,old,new in planned:
            if old != new:
                backup = path.with_suffix('.py.before_native')
                if not backup.exists(): backup.write_text(old)
                path.write_text(new)
            records.append({'path':str(path.relative_to(root)),
                            'sha256':hashlib.sha256(new.encode()).hexdigest()})
        importlib.invalidate_caches()
        return records
    
    STYLEGAN_NATIVE_PATCHES = install_native_stylegan_ops(LOCAL_REPO)
    print('StyleGAN backend:', STYLEGAN_OPS_BACKEND, '| patched files:', len(STYLEGAN_NATIVE_PATCHES))
    


In [ ]:
if NEED_EVALUATOR:
    from PIL import Image
    if not hasattr(Image,"ANTIALIAS"): Image.ANTIALIAS=Image.Resampling.LANCZOS
    import sys, torch
    sys.path.insert(0, LOCAL_REPO)
    from hair_swap import HairFast, get_parser
    hair_fast = HairFast(get_parser().parse_args([]))
    print('HairFastGAN พร้อม')
    
    import numpy as np,pandas as pd
    from PIL import Image,ImageOps,ImageFilter,ImageDraw
    import torch.nn.functional as F
    from torch.utils.checkpoint import checkpoint
    from models.Net import get_segmentation as segment
    from IPython.display import display
    b=hair_fast.blend;enc=b.blending_encoder;G=b.net.generator;pp=b.post_process;align=hair_fast.align
    for module in [enc,G,pp,align]:module.eval().requires_grad_(False)


In [ ]:
if NEED_EVALUATOR:
    runtime_baseS={}
    def cpu(x):
        if torch.is_tensor(x):
            return x.detach().cpu().clone()
        if isinstance(x, dict):
            return {k: cpu(v) for k, v in x.items()}
        if isinstance(x, list):
            return [cpu(v) for v in x]
        if isinstance(x, tuple):
            return tuple((cpu(v) for v in x))
        return copy.deepcopy(x)
    
    def gpu(x):
        if torch.is_tensor(x):
            return x.to('cuda').clone()
        if isinstance(x, dict):
            return {k: gpu(v) for k, v in x.items()}
        if isinstance(x, list):return [gpu(v) for v in x]
        if isinstance(x, tuple):return tuple(gpu(v) for v in x)
        return x
    
    def resize(x):
        return F.interpolate(x, size=(256, 256), mode='bilinear', align_corners=False)
    
    def saveim(x, p):
        a = x.detach().float().cpu()[0].clamp(0, 1).permute(1, 2, 0).numpy()
        Image.fromarray((a * 255).round().astype('uint8')).save(p)
    
    def lab(x):
        x = x.float().clamp(0, 1)
        lin = torch.where(x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055).pow(2.4))
        M = x.new_tensor([[0.4124564, 0.3575761, 0.1804375], [0.2126729, 0.7151522, 0.072175], [0.0193339, 0.119192, 0.9503041]])
        xyz = torch.einsum('ij,bjhw->bihw', M, lin) / x.new_tensor([0.95047, 1.0, 1.08883])[None, :, None, None]
        f = torch.where(xyz > 0.008856, xyz.clamp_min(1e-08).pow(1 / 3), 7.787 * xyz + 16 / 116)
        return torch.stack([116 * f[:, 1] - 16, 500 * (f[:, 0] - f[:, 1]), 200 * (f[:, 1] - f[:, 2])], 1)
    
    def render(s, d):
        S = torch.cat([d['S'][:, :6], s], 1)
        return G([S], input_is_latent=True, return_latents=False, start_layer=4, end_layer=8, layer_in=d['F'], randomize_noise=False)[0]
    
    def finish(raw, d):
        sf, ff = pp(d['I1'], b.downsample_256(raw))
        return G([sf], input_is_latent=True, return_latents=False, start_layer=5, end_layer=8, layer_in=ff, randomize_noise=False)[0]
    
    def detail(x):
        y = x.mean(1, keepdim=True)
        mu = F.avg_pool2d(y, 9, 1, 4)
        v = F.avg_pool2d((y - mu).square(), 9, 1, 4)
        return (y - mu) / (v + 0.001).sqrt()
    
    def unit(raw):return resize(((raw+1)/2).clamp(0,1))
    def read_mask(p):
        with Image.open(p) as im:
            if im.size!=(256,256):raise ValueError('Mask size mismatch')
            a=np.asarray(im.convert('L'))>127
        return torch.from_numpy(a.copy())[None,None]
    
    def load_case(k):
        d=torch.load(SOURCE/'cache'/(k+'.pt'),map_location='cpu',weights_only=True)
        folder=STUDY_ROOT/'trial_A/inputs'/k
        for name in ['eval_core','eval_refmask','target','protected']:
            mask=read_mask(folder/(name+'.png'))
            if name in ['target','protected']:
                assert torch.equal(d[name].bool(),mask), (k,name)
            d[name]=mask
        for name in ['core','refmask']:d[name]=d[name].bool()
        for name in ['eval_core','eval_refmask']:assert d[name].sum()>=100
        return d
    
    def quantile_color(im,ref,mask,refmask):
        l=lab(im)[0,:,mask[0,0].bool()];r=lab(ref)[0,:,refmask[0,0].bool()]
        q=torch.linspace(.1,.9,9,device=im.device)
        diff=(torch.quantile(l,q,dim=1)-torch.quantile(r,q,dim=1)).abs()
        return diff[:,0].mean(),diff[:,1:].mean()
    
    def masked_mae(a,b,m):return ((a-b).abs()*m.float()).sum()/(a.shape[1]*m.sum().clamp_min(1))
    
    def shape_metrics(im,d):
        labels=segment(align.to_bisenet(im))
        pred=F.interpolate((labels==13).float().reshape(1,1,*labels.shape[-2:]),(256,256),mode='nearest')>.5
        target=d['target']
        return dict(target_mask_IoU=float((pred&target).sum()/(pred|target).sum().clamp_min(1)),
                    target_missing_fraction=float((target&~pred).sum()/target.sum().clamp_min(1)))
    
    
    # Frozen differentiable evaluators: reuse exact parser weights, load original ArcFace.
    from models.CtrlHair.external_code.face_parsing.my_parsing_util import FaceParsing, FaceParsing_tensor
    from losses.pp_losses import Backbone
    from huggingface_hub import hf_hub_download
    id_path=Path('pretrained_models/ArcFace/ir_se50.pth')
    if not id_path.is_file():
        downloaded=hf_hub_download(repo_id='AIRI-Institute/HairFastGAN',filename=str(id_path))
        id_path.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(downloaded,id_path)
    id_net=Backbone(input_size=112,num_layers=50,drop_ratio=.6,mode='ir_se')
    id_net.load_state_dict(torch.load(id_path,map_location='cpu',weights_only=True),strict=True)
    id_net=id_net.cuda().eval().requires_grad_(False)
    FaceParsing_tensor.parsing_img()
    parser_net=FaceParsing.bise_net
    if parser_net is None:raise RuntimeError('Original BiSeNet is not initialized')
    parser_net.eval().requires_grad_(False)
    assert FaceParsing.label_list[17]=='hair' # raw parser label=17; remapped target label=13
    
    def id_features(im):
        x=F.interpolate(im,size=(256,256),mode='bilinear',align_corners=False)*2-1
        x=F.adaptive_avg_pool2d(x[:,:,35:223,32:220],(112,112))
        feats=id_net(x)
        return F.normalize(feats,dim=1)
    
    def constraint_terms(im,d,fx):
        # Do NOT call get_segmentation: its no_grad + argmax would sever training gradients.
        logits=checkpoint(lambda x:parser_net(align.to_bisenet(x))[0],im,use_reentrant=False) if torch.is_grad_enabled() else parser_net(align.to_bisenet(im))[0]
        other=torch.cat((logits[:,:17],logits[:,18:]),1)
        binary=logits[:,17:18]-torch.logsumexp(other,dim=1,keepdim=True)
        target=F.interpolate(d['target'].float(),size=binary.shape[-2:],mode='nearest')
        if target.sum()==0 or (1-target).sum()==0:raise RuntimeError('Degenerate shape target')
        prob=binary.sigmoid()
        bce=F.binary_cross_entropy_with_logits(binary,target,reduction='none')
        balanced=.5*((bce*target).sum()/target.sum()+(bce*(1-target)).sum()/(1-target).sum())
        dice=1-(2*(prob*target).sum()+1)/(prob.sum()+target.sum()+1)
        embedding=checkpoint(id_features,im,use_reentrant=False) if torch.is_grad_enabled() else id_features(im)
        identity=(1-(embedding*fx['face_embedding']).sum(1)).mean()
        return identity,balanced+dice,balanced,dice
    
    def to_size(x,size):
        return F.interpolate(x,size=(size,size),mode='bilinear',align_corners=False)
    
    def detail_terms(full,d,fx):
        values={}
        for size in [256,512]:
            x=to_size(full,size)
            teacher=fx['pre'] if size==256 else fx['pre_512']
            mask=F.interpolate(d['eval_core'].float(),size=(size,size),mode='nearest')
            inner=-F.max_pool2d(-mask,9,1,4)
            if inner.sum()<64:raise RuntimeError('Insufficient interior hair pixels at '+str(size))
            values[size]=masked_mae(detail(x),detail(teacher),inner)
        return values
    
    
    from scipy.ndimage import binary_erosion,binary_dilation
    from skimage.metrics import structural_similarity
    _basic_shape_metrics=shape_metrics
    def shape_metrics(im,d):
        result=_basic_shape_metrics(im,d)
        labels=segment(align.to_bisenet(im));p=F.interpolate((labels==13).float().reshape(1,1,*labels.shape[-2:]),(256,256),mode='nearest')[0,0].cpu().numpy()>.5
        t=d['target'][0,0].cpu().numpy()>.5
        result['target_Dice']=float(2*(p&t).sum()/max(p.sum()+t.sum(),1))
        a=p^binary_erosion(p);bnd=t^binary_erosion(t)
        prec=(a&binary_dilation(bnd,iterations=2)).sum()/max(a.sum(),1)
        rec=(bnd&binary_dilation(a,iterations=2)).sum()/max(bnd.sum(),1)
        result['boundary_F1_2px']=float(2*prec*rec/max(prec+rec,1e-12))
        return result
    
    def extra_metrics(im,d):
        x=im[0].permute(1,2,0).cpu().numpy();y=d['face'][0].permute(1,2,0).cpu().numpy()
        mask=d['protected'][0,0].cpu().numpy()>.5
        _,ss=structural_similarity(x,y,data_range=1.,channel_axis=2,full=True)
        safe=binary_erosion(mask,iterations=3)
        if not safe.any():raise ValueError('No protected SSIM interior')
        mse=float(((x-y)**2)[mask].mean())
        return dict(protected_PSNR=float(-10*np.log10(max(mse,1e-12))),protected_SSIM=float(ss[safe].mean()),
            any_channel_255_fraction=float((im[0,:,d['eval_core'][0,0]].max(0).values>=1-0.5/255).float().mean()))


## ฟังก์ชันต้นฉบับ — คัดลอกทั้ง cell โดยไม่เปลี่ยน prompt, mask หรือการผสมภาพ

In [ ]:
# P1–P5 เน้นคนละคุณสมบัติ; P6 รวมแบบสมดุล ไม่ต่อข้อความยาวทั้งหมด
PROMPT_SET_VERSION = 'preserve_input_color_v3'
BASE_PROMPT = 'Photorealistic portrait. Preserve the input hair hue, saturation, brightness, highlights and shadows. Keep the same face, haircut and background. '
COMMON_NEGATIVE = 'gray haze, washed-out hair, color shift, overexposure, plastic hair, wig, halo, changed haircut, deformed face'
FOCUS_PHRASES = {
    'prompt1_texture': 'Refine fine irregular strand texture within existing locks. Keep their layout and gloss.',
    'prompt2_color': 'Retain the existing hair color and natural tonal variation. No recoloring or added gray veil.',
    'prompt3_sharpness': 'Resolve fine individual hair fibers and narrow strand separation without sharpening halos.',
    'prompt4_blend': 'Refine only the hair boundary into fine tapered strands. Preserve interior texture.',
    'prompt5_lighting': 'Keep the existing light direction and highlight intensity. Refine local strand shading without global relighting.',
    'prompt6_combined': 'Refine fine strand texture, fiber separation and tapered hair edges. Preserve local shading and gloss.'
}
REFINE_PROMPTS = [{'tag':tag,'prompt':BASE_PROMPT+phrase,'negative_prompt':COMMON_NEGATIVE}
                  for tag,phrase in FOCUS_PHRASES.items()]
# Reference เข้าขั้น HairFastGAN เท่านั้น; diffusion นี้เห็น A ไม่ได้เห็น reference โดยตรง
DEFAULT_PROMPT_TAG = 'prompt6_combined'
INPAINT_STEPS = 20
INPAINT_STRENGTH = 0.5  # จุดเริ่มทดลอง ไม่ใช่ค่าที่รับรองว่าดีที่สุด
INPAINT_GUIDANCE = 7.5
DEFAULT_SEED = 3407
HAIRFAST_SEED = 3407
FACE_MARGIN_PX = 3      # pixels ที่ resolution ของภาพ A
BLUR_RADIUS = 2.0
# Fixed value range, ไม่ normalize ด้วย min/max ต่อภาพเพราะจะทำให้สีเปลี่ยน
# official HairFast output ใช้ภาพในช่วง [0,1]; ถ้า repo ที่ใช้ต่างไปให้ตรวจและเปลี่ยน
OUTPUT_TENSOR_RANGE = 'zero_one'
PROMPTS = {p['tag']: p for p in REFINE_PROMPTS}


def segment_labels(image):
    inputs = seg_processor(images=image.convert('RGB'), return_tensors='pt').to(DEVICE)
    with torch.inference_mode():
        logits = seg_model(**inputs).logits
        logits = torch.nn.functional.interpolate(logits, size=image.size[::-1], mode='bilinear', align_corners=False)
    return logits.argmax(1)[0].cpu().numpy().astype(np.uint8)


def build_masks(labels, margin_px=FACE_MARGIN_PX, blur_radius=BLUR_RADIUS):
    hair = labels == HAIR_ID
    face = labels == FACE_ID
    if not hair.any():
        raise ValueError('ไม่พบเส้นผมในภาพ กรุณาตรวจภาพหรือ segmentation')
    if not face.any():
        raise ValueError('ไม่พบใบหน้า ไม่สามารถใช้โหมดปกป้องหน้าได้')
    face_img = Image.fromarray(face.astype(np.uint8) * 255)
    protected = np.asarray(face_img.filter(ImageFilter.MaxFilter(2 * margin_px + 1))) > 0
    hair_img = Image.fromarray(hair.astype(np.uint8) * 255)
    soft = np.asarray(hair_img.filter(ImageFilter.GaussianBlur(blur_radius)), dtype=np.float32) / 255
    # C แก้เฉพาะผมที่ segmentation เลือก ไม่แก้ background/face แม้ขอบ blur ล้ำ
    allowed = hair & ~protected
    alpha = soft * allowed
    if not np.any(alpha > 0):
        raise ValueError('พื้นที่ผมหลังกันหน้าว่าง ต้องตรวจ mask')
    return {'hair': hair, 'face': face, 'protected': protected,
            'alpha': alpha.astype(np.float32), 'raw_mask': Image.fromarray(np.rint(soft * 255).astype(np.uint8))}


def composite_protected(base, edited, alpha):
    if base.size != edited.size or np.asarray(alpha).shape != (base.height, base.width):
        raise ValueError('ภาพและ alpha ต้องมีขนาดตรงกัน')
    a = np.clip(np.asarray(alpha, dtype=np.float32), 0, 1)[..., None]
    original = np.array(base.convert('RGB'))
    changed = np.asarray(edited.convert('RGB'), dtype=np.float32)
    output = np.rint(original * (1 - a) + changed * a).clip(0, 255).astype(np.uint8)
    output[a[..., 0] == 0] = original[a[..., 0] == 0]
    return Image.fromarray(output)


def mask_overlay(image, masks):
    a = np.asarray(image.convert('RGB')).copy().astype(float)
    for mask, rgb in [(masks['alpha'] > 0, (0, 255, 0)), (masks['protected'], (255, 0, 0))]:
        a[mask] = 0.55 * a[mask] + 0.45 * np.array(rgb)
    return Image.fromarray(a.astype(np.uint8))


def padded(image, target=512, mask=False):
    scale = target / max(image.size)
    size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
    resized = image.resize(size, Image.Resampling.NEAREST if mask else Image.Resampling.LANCZOS)
    canvas = Image.new(image.mode, (target, target), 0)
    x, y = (target-size[0])//2, (target-size[1])//2
    canvas.paste(resized, (x, y))
    return canvas, (x, y, x+size[0], y+size[1])


def tensor_to_pil(value, value_range='zero_one'):
    if isinstance(value, Image.Image):
        return value.convert('RGB')
    if torch.is_tensor(value):
        t = value.detach().cpu()
        if t.ndim == 4 and t.shape[0] == 1:
            t = t[0]
        if t.ndim != 3 or t.shape[0] != 3:
            raise ValueError(f'Unexpected tensor shape: {tuple(t.shape)}')
        a = t.permute(1, 2, 0).numpy()
    else:
        a = np.asarray(value)
    if a.dtype == np.uint8:
        return Image.fromarray(a).convert('RGB')
    if value_range == 'minus_one_one':
        a = (a + 1) / 2
    if not np.isfinite(a).all() or a.min() < -0.02 or a.max() > 1.02:
        raise ValueError('Tensor range ไม่ตรง config: ตรวจ OUTPUT_TENSOR_RANGE ห้าม normalize อัตโนมัติ')
    return Image.fromarray(np.rint(a.clip(0, 1) * 255).astype(np.uint8)).convert('RGB')


def resolve_prompt_configs(overrides=None):
    # คัดลอกต่อคำขอ ไม่เปลี่ยน preset ส่วนกลาง
    configs = {tag:dict(cfg) for tag,cfg in PROMPTS.items()}
    if overrides is None:
        return configs
    if not isinstance(overrides, dict) or len(overrides) > len(PROMPTS):
        raise ValueError('prompt_overrides ต้องเป็น object ตามชื่อ prompt ที่รองรับ')
    for tag, change in overrides.items():
        if tag not in configs or not isinstance(change, dict):
            raise ValueError('ชื่อ prompt หรือรูปแบบ override ไม่ถูกต้อง')
        if set(change) - {'prompt','negative_prompt'}:
            raise ValueError('แก้ได้เฉพาะ prompt และ negative_prompt')
        for field, text in change.items():
            if not isinstance(text, str) or len(text) > 4000:
                raise ValueError('ข้อความ prompt ต้องเป็น string ไม่เกิน 4000 ตัวอักษร')
            if field == 'prompt' and not text.strip():
                raise ValueError('prompt ห้ามว่าง')
            configs[tag][field] = text
    return configs


def parse_prompt_changes(prompt_tag, mode, prompt=None, negative_prompt=None, prompt_overrides=None):
    import json
    if prompt_tag not in PROMPTS:
        raise ValueError('ไม่พบ prompt_tag')
    if prompt_overrides is not None:
        if not isinstance(prompt_overrides, str) or len(prompt_overrides) > 60000:
            raise ValueError('prompt_overrides ต้องเป็น JSON string ไม่เกิน 60000 ตัวอักษร')
        try:
            changes = json.loads(prompt_overrides)
        except (ValueError, TypeError):
            raise ValueError('prompt_overrides เป็น JSON ไม่ถูกต้อง')
        resolve_prompt_configs(changes)
        if not isinstance(changes, dict):
            raise ValueError('prompt_overrides ต้องเป็น JSON object')
    else:
        changes = {}
    if prompt is not None or negative_prompt is not None:
        if mode in ('all_prompts', 'all_prompts_unprotected', 'all_prompts_joint_controlled','all_prompts_joint_unprotected','all_prompts_joint_controlled_noeyes','all_prompts_joint_unprotected_noeyes','all_prompts_joint_controlled_eyerefine','all_prompts_joint_unprotected_eyerefine'):
            raise ValueError('all_prompts ให้ใช้ prompt_overrides เพื่อระบุข้อความแยกแต่ละชุด')
        if prompt_tag in changes:
            raise ValueError('ส่งข้อความซ้ำกัน: เลือกใช้ prompt/negative_prompt หรือ prompt_overrides')
        changes[prompt_tag] = {}
        if prompt is not None:
            changes[prompt_tag]['prompt'] = prompt
        if negative_prompt is not None:
            changes[prompt_tag]['negative_prompt'] = negative_prompt
    if mode == 'baseline' and changes:
        raise ValueError('baseline ไม่ใช้ diffusion prompt ให้เลือก protected/abc/all_prompts')
    if mode not in ('all_prompts', 'all_prompts_unprotected', 'all_prompts_joint_controlled','all_prompts_joint_unprotected','all_prompts_joint_controlled_noeyes','all_prompts_joint_unprotected_noeyes','all_prompts_joint_controlled_eyerefine','all_prompts_joint_unprotected_eyerefine') and set(changes) - {prompt_tag}:
        raise ValueError('โหมดนี้แก้ได้เฉพาะ prompt_tag ที่เลือก')
    resolve_prompt_configs(changes)
    return changes


def resolve_inpaint_settings(strength=None, steps=None, guidance=None):
    import math
    strength = float(INPAINT_STRENGTH if strength is None else strength)
    steps = INPAINT_STEPS if steps is None else steps
    guidance = float(INPAINT_GUIDANCE if guidance is None else guidance)
    if isinstance(steps, bool) or not isinstance(steps, (int, np.integer)) or not 1 <= steps <= 100:
        raise ValueError('steps ต้องเป็นจำนวนเต็ม 1–100')
    steps = int(steps)
    if not math.isfinite(strength) or not 0 < strength <= 1:
        raise ValueError('strength ต้องมากกว่า 0 และไม่เกิน 1')
    if not math.isfinite(guidance) or not 1 <= guidance <= 20:
        raise ValueError('guidance ต้องอยู่ในช่วง 1–20')
    if int(steps * strength) < 1:
        raise ValueError('steps × strength ต้องไม่น้อยกว่า 1')
    return {'strength':strength, 'steps':steps, 'guidance':guidance}


def refine_once(
    base, masks, prompt_config, seed, protected,
    strength=None, steps=None, guidance=None, trace_dir=None
):
    validate_joint_prompt_lengths({prompt_config.get('tag','custom'):prompt_config})
    settings = resolve_inpaint_settings(strength, steps, guidance)
    image512, box = padded(base)

    if protected:
        alpha_img = Image.fromarray(
            np.rint(masks['alpha'] * 255).astype(np.uint8)
        )
        mask512, _ = padded(alpha_img, mask=True)

        protect512, _ = padded(
            Image.fromarray(
                masks['protected'].astype(np.uint8) * 255
            ),
            mask=True
        )
        protect512 = np.asarray(
            protect512.filter(ImageFilter.MaxFilter(3))
        ) > 0

        m = np.array(mask512)
        m[protect512] = 0
        mask512 = Image.fromarray(m)
    else:
        mask512, _ = padded(masks['raw_mask'], mask=True)

    if not (np.asarray(mask512) > 127).any():
        raise ValueError('mask เล็กเกินไปหลัง resize เป็น 512')

    # ===== เพิ่ม: ตรวจภาพและ mask ก่อนเข้า Diffusion =====
    from IPython.display import display

    print("\nPrompt:", prompt_config.get('tag', 'custom'))
    print("ข้อความที่ส่งจริง:", prompt_config['prompt'])
    print("Negative:", prompt_config['negative_prompt'])
    print("ค่าที่ใช้:", settings, "| Seed:", seed)
    print("ปกป้องหน้าใน refine_once:", protected)
    # Joint ส่ง protected=False ตรงนี้ แล้วผสมภาพใน refine_joint ภายหลัง

    image_debug = image512.convert("RGB")
    mask_debug = mask512.convert("L")

    # สีแดงทับบริเวณ mask ที่มีค่ามากกว่า 127
    selected = np.asarray(mask_debug) > 127
    overlay = np.asarray(image_debug).copy()
    overlay[selected] = (
        overlay[selected].astype(np.float32) * 0.55
        + np.array([255, 0, 0], dtype=np.float32) * 0.45
    ).astype(np.uint8)

    width, height = image_debug.size
    panel = Image.new("RGB", (width * 3, height))
    panel.paste(image_debug, (0, 0))
    panel.paste(mask_debug.convert("RGB"), (width, 0))
    panel.paste(Image.fromarray(overlay), (width * 2, 0))

    print("ซ้าย: ภาพเข้าโมเดล | กลาง: mask | ขวา: พื้นที่แก้สีแดง")
    display(panel)
    # ===== จบส่วนตรวจ =====

    generator = torch.Generator(
        device=DEVICE
    ).manual_seed(int(seed))

    with torch.inference_mode():
        result = inpaint_pipe(
            image=image512,
            mask_image=mask512,
            prompt=prompt_config['prompt'],
            negative_prompt=prompt_config['negative_prompt'],
            num_inference_steps=settings["steps"],
            strength=settings["strength"],
            guidance_scale=settings["guidance"],
            generator=generator
        )

    flags = getattr(result, 'nsfw_content_detected', None)
    if flags is not None and any(flags):
        raise ValueError('ผลภาพถูกตัวกรองของโมเดลระงับ')

    edited = (
        result.images[0]
        .convert('RGB')
        .crop(box)
        .resize(base.size, Image.Resampling.LANCZOS)
    )

    if trace_dir is not None:
        trace_dir = Path(trace_dir)
        trace_dir.mkdir(parents=True, exist_ok=True)
        image512.save(trace_dir/'diffusion_input_512.png')
        mask512.save(trace_dir/'diffusion_mask_512.png')
        result.images[0].convert('RGB').save(trace_dir/'diffusion_raw_512.png')
        edited.save(trace_dir/'diffusion_raw_before_blend.png')
        # Resize-only control isolates resolution loss from the generated changes.
        image512.crop(box).resize(base.size, Image.Resampling.LANCZOS).save(trace_dir/'resize_only_control.png')

    return (
        composite_protected(base, edited, masks['alpha'])
        if protected
        else edited
    )


# Single-pass joint hair + face study: same six prompts/noise/mask in both modes.
JOINT_MODES = ('all_prompts_joint_controlled','all_prompts_joint_unprotected','all_prompts_joint_controlled_noeyes','all_prompts_joint_unprotected_noeyes','all_prompts_joint_controlled_eyerefine','all_prompts_joint_unprotected_eyerefine')
JOINT_FACE_BLEND = 0.35
JOINT_TRANSITION_PX = 24  # measured at 512px short side; scaled with image size
JOINT_MARGIN_PX = 12
JOINT_INSTRUCTION = BASE_PROMPT
JOINT_FOCUS_PHRASES = dict(FOCUS_PHRASES)

JOINT_CONTROLLED_MODES = ('all_prompts_joint_controlled', 'all_prompts_joint_controlled_noeyes', 'all_prompts_joint_controlled_eyerefine')
JOINT_EYE_MODES = ('all_prompts_joint_controlled', 'all_prompts_joint_unprotected','all_prompts_joint_controlled_eyerefine','all_prompts_joint_unprotected_eyerefine')

def resolve_joint_settings(face_blend=JOINT_FACE_BLEND):
    import math
    face_blend = float(face_blend)
    if not math.isfinite(face_blend) or not 0 <= face_blend <= 1:
        raise ValueError('face_blend ต้องอยู่ในช่วง 0–1')
    return {'face_blend':face_blend}


def joint_prompt_configs(configs):
    # Keep P1–P6 axes; replace the hair-only preamble with a short joint preamble.
    # Compact P6 to avoid swallowing its five axes in a long prompt.
    result = {}
    for tag,cfg in configs.items():
        body = JOINT_FOCUS_PHRASES[tag] if cfg['prompt'] == PROMPTS[tag]['prompt'] else cfg['prompt']
        result[tag] = {**cfg, 'prompt':JOINT_INSTRUCTION + body}
    return result


def validate_joint_prompt_lengths(configs):
    # Validate using the tokenizer actually loaded in Colab, not character counts.
    tokenizer = getattr(inpaint_pipe, 'tokenizer', None)
    if tokenizer is None:
        return
    limit = tokenizer.model_max_length
    for tag,cfg in configs.items():
        for field in ('prompt','negative_prompt'):
            count = len(tokenizer(cfg[field], truncation=False)['input_ids'])
            if count > limit:
                raise ValueError(f'{tag} {field}: {count} tokens เกิน {limit}; ย่อข้อความเพื่อไม่ให้ส่วนท้ายถูกตัด')


# รักษาตาจากภาพ A ใน Joint ทั้งสองโหมด (ปิดได้เพื่อเทียบผลเดิม)
# Eye protection เลือกตาม mode ของแต่ละคำขอ ไม่ใช้ global switch
JOINT_EYE_MARGIN_PX = 3     # ที่ด้านสั้น 512: ครอบขอบเปลือกตา
JOINT_EYE_FEATHER_PX = 6    # ไล่ขอบออกด้านนอก ไม่ลดการปกป้องแกนกลาง
_EYE_LANDMARK_MODEL = None


def detect_eye_landmarks(base):
    """ใช้ face-alignment ที่ติดตั้งไว้แล้ว; โหลดครั้งแรก ใช้ CPU ลด VRAM."""
    global _EYE_LANDMARK_MODEL
    import face_alignment
    import inspect
    if _EYE_LANDMARK_MODEL is None:
        kwargs = dict(device='cpu', flip_input=False)
        if 'compile' in inspect.signature(face_alignment.FaceAlignment).parameters:
            kwargs['compile'] = False
        _EYE_LANDMARK_MODEL = face_alignment.FaceAlignment(
            face_alignment.LandmarksType.TWO_D, **kwargs)
    with torch.inference_mode():
        predictions = _EYE_LANDMARK_MODEL.get_landmarks(np.asarray(base.convert('RGB')))
    if predictions is None or len(predictions) != 1:
        raise ValueError('Eye protection: ต้องตรวจพบใบหน้าเดียวในภาพ A กรุณาใช้ภาพหน้าชัดเจน')
    points = np.asarray(predictions[0], dtype=np.float32)
    if points.shape != (68, 2) or not np.isfinite(points).all():
        raise ValueError('Eye protection: landmarks ไม่ถูกต้อง')
    return points


def build_eye_protection(size, landmarks):
    """eye_keep=1 คืน A เต็มที่; 0 ใช้ Joint ตามเดิม; ขอบมีช่วงเปลี่ยนที่จำกัด."""
    from scipy.ndimage import distance_transform_edt
    w, h = size
    canvas = Image.new('L', (w, h), 0)
    draw = ImageDraw.Draw(canvas)
    for eye in (landmarks[36:42], landmarks[42:48]):
        if np.ptp(eye[:, 0]) < 2 or (eye < 0).any() or (eye[:, 0] >= w).any() or (eye[:, 1] >= h).any():
            raise ValueError('Eye protection: ตำแหน่งตาอยู่นอกภาพหรือเล็กเกินไป')
        draw.polygon([tuple(map(float, p)) for p in eye], fill=255)
    scale = min(w, h) / 512.0
    margin = max(0, round(JOINT_EYE_MARGIN_PX * scale))
    if margin:
        canvas = canvas.filter(ImageFilter.MaxFilter(2 * margin + 1))
    core = np.asarray(canvas) > 0
    feather = max(1.0, JOINT_EYE_FEATHER_PX * scale)
    x = np.clip(distance_transform_edt(~core) / feather, 0, 1)
    keep = (1 - x*x*(3-2*x)).astype(np.float32)
    keep[core] = 1.0
    return {'eye_core': core, 'eye_keep': keep}


def build_joint_masks(masks, face_blend=JOINT_FACE_BLEND, base=None, mode='all_prompts_joint_controlled'):
    if mode not in JOINT_MODES:
        raise ValueError('ไม่ใช่โหมด joint')
    from scipy.ndimage import binary_fill_holes, distance_transform_edt
    face_blend = resolve_joint_settings(face_blend)['face_blend']
    h,w = masks['face'].shape
    scale = min(h,w) / 512.0
    margin = max(1, round(JOINT_MARGIN_PX * scale))
    transition = max(1.0, JOINT_TRANSITION_PX * scale)
    # Fill enclosed holes (e.g. glasses) and close small segmentation gaps at hairline.
    union = masks['hair'] | masks['face']
    radius = max(1, round(3 * scale))
    u = Image.fromarray(union.astype(np.uint8)*255)
    closed = np.asarray(u.filter(ImageFilter.MaxFilter(2*radius+1)).filter(
        ImageFilter.MinFilter(2*radius+1))) > 0
    core = binary_fill_holes(union | closed)
    core_img = Image.fromarray(core.astype(np.uint8)*255)
    support = np.asarray(core_img.filter(ImageFilter.MaxFilter(2*margin+1))) > 0
    # Feather only the outer perimeter; never cut the shared face/hair boundary.
    distance = distance_transform_edt(np.pad(support, 1))[1:-1,1:-1]
    x = np.clip(distance / margin, 0, 1)
    outer_alpha = (x*x*(3-2*x)).astype(np.float32)
    outer_alpha[core] = 1.0
    # Smoothly reduce generated-image weight towards the interior of the face.
    face = binary_fill_holes(masks['face']) & ~masks['hair']
    distance_face = distance_transform_edt(np.pad(face, 1))[1:-1,1:-1]
    x = np.clip(distance_face / transition, 0, 1)
    face_weight = (x*x*(3-2*x)).astype(np.float32)
    controlled_alpha = outer_alpha * (1 - (1-face_blend)*face_weight)
    if mode in JOINT_EYE_MODES:
        if base is not None:
            if base.size != (w, h):
                raise ValueError('Eye protection: ภาพ A และ mask ขนาดไม่ตรงกัน')
            masks.update(build_eye_protection(base.size, detect_eye_landmarks(base)))
        if 'eye_keep' not in masks:
            raise ValueError('Eye protection: ต้องส่งภาพ A เพื่อหาตำแหน่งตาก่อน')
        eye_keep, eye_core = masks['eye_keep'], masks['eye_core']
    else:
        eye_keep = np.zeros((h, w), dtype=np.float32)
        eye_core = np.zeros((h, w), dtype=bool)
    return {'eye_keep':eye_keep, 'eye_core':eye_core,
            'raw_mask':Image.fromarray(support.astype(np.uint8)*255),
            'core':core, 'support':support, 'face_weight':face_weight,
            'outer_alpha':outer_alpha, 'controlled_alpha':controlled_alpha,
            'margin_px_actual':margin, 'transition_px_actual':transition}


def joint_alpha(joint, mode):
    if mode not in JOINT_MODES:
        raise ValueError('ไม่ใช่โหมด joint')
    alpha = joint['controlled_alpha'] if mode in JOINT_CONTROLLED_MODES else joint['outer_alpha']
    return alpha * (1 - joint['eye_keep']) if mode in JOINT_EYE_MODES else alpha.copy()



JOINT_EYE_REFINE_MODES = ('all_prompts_joint_controlled_eyerefine','all_prompts_joint_unprotected_eyerefine')
EYE_REFINE_STRENGTH = 0.20  # ความแรงการสร้างรายละเอียดตา แยกจาก Strength ของ Joint
EYE_REFINE_BLEND = 0.35     # น้ำหนักตาที่ AI ปรับ; ที่เหลือใช้ A
EYE_REFINE_STEPS = 30
EYE_REFINE_GUIDANCE = 4.0
EYE_REFINE_PROMPT = ('Photorealistic eyes, subtly refine existing eye detail. Preserve eye shape, '
    'iris size, iris color, gaze direction and eyelids. Natural texture and soft lighting.')
EYE_REFINE_NEGATIVE = ('different eye shape, different iris color, changed gaze, enlarged eyes, '
    'asymmetric pupils, extra pupils, heavy makeup, oversharpening, artifacts')


def refine_eyes_from_a(base, joint, seed):
    # Cache ต่อคำขอ: ทั้ง P1–P6 ใช้ตาปรับชุดเดียวกัน ไม่สร้างหกรอบ
    import hashlib
    key=(hashlib.sha256(base.tobytes()).hexdigest(), int(seed), EYE_REFINE_STRENGTH,
         EYE_REFINE_BLEND,EYE_REFINE_STEPS,EYE_REFINE_GUIDANCE,
         EYE_REFINE_PROMPT,EYE_REFINE_NEGATIVE)
    if joint.get('_eye_cache_key')==key:
        return joint['_eye_cache_image']
    if not 0<=EYE_REFINE_BLEND<=1:
        raise ValueError('EYE_REFINE_BLEND ต้องอยู่ในช่วง 0–1')
    if EYE_REFINE_BLEND==0:
        return base.copy()
    ys,xs=np.where(joint['eye_keep']>0)
    if len(xs)==0: raise ValueError('ไม่พบพื้นที่ตาสำหรับปรับรายละเอียด')
    # ครอปตาคู่พร้อมบริบทจาก A เพื่อไม่ย่อทั้งหน้าแล้วสูญรายละเอียดตา
    width=xs.max()-xs.min()+1; height=ys.max()-ys.min()+1
    padx=max(16,round(width*.25));pady=max(16,round(height*1.5))
    box=(max(0,int(xs.min()-padx)),max(0,int(ys.min()-pady)),
         min(base.width,int(xs.max()+1+padx)),min(base.height,int(ys.max()+1+pady)))
    x0,y0,x1,y1=box
    crop=base.crop(box)
    raw_mask=Image.fromarray((joint['eye_keep'][y0:y1,x0:x1]>0).astype(np.uint8)*255)
    cfg={'tag':'eye_detail_from_A','prompt':EYE_REFINE_PROMPT,'negative_prompt':EYE_REFINE_NEGATIVE}
    validate_joint_prompt_lengths({'eye_detail_from_A':cfg})
    edited=refine_once(crop,{'raw_mask':raw_mask},cfg,seed,False,
        strength=EYE_REFINE_STRENGTH,steps=EYE_REFINE_STEPS,guidance=EYE_REFINE_GUIDANCE)
    canvas=base.copy();canvas.paste(edited,box[:2])
    # น้ำหนักคงที่ก่อนประกอบด้วย eye_keep อีกครั้ง: ป้องกัน feather ซ้ำ
    candidate=composite_protected(base,canvas,np.full((base.height,base.width),EYE_REFINE_BLEND,dtype=np.float32))
    joint['_eye_cache_key']=key;joint['_eye_cache_image']=candidate
    return candidate

def refine_joint(base, masks, joint, prompt_config, seed, mode, **settings):
    # Exactly one diffusion call; both modes generate with the same full joint mask.
    inference_masks = {**masks, 'raw_mask':joint['raw_mask']}
    generated = refine_once(base, inference_masks, prompt_config, seed, False, **settings)
    # Eye modes คืนตาจาก A; noeyes ไม่มีการตัดตาออกจากน้ำหนัก Blend
    result = composite_protected(base, generated, joint_alpha(joint, mode))
    if mode in JOINT_EYE_REFINE_MODES:
        eye_image = refine_eyes_from_a(base, joint, seed)
        # เริ่มจากน้ำหนัก Joint ปกติ เพื่อไม่ feather ตา A ซ้ำ
        normal_alpha = joint['controlled_alpha'] if mode in JOINT_CONTROLLED_MODES else joint['outer_alpha']
        result = composite_protected(base, generated, normal_alpha)
        result = composite_protected(result, eye_image, joint['eye_keep'])
    return result


def joint_metadata(mode, face_blend):
    if mode not in JOINT_MODES:
        return {}
    blend = resolve_joint_settings(face_blend)['face_blend']
    return {'joint_refinement': {
        'eye_protection':mode in JOINT_EYE_MODES and mode not in JOINT_EYE_REFINE_MODES,
        'eye_refinement':mode in JOINT_EYE_REFINE_MODES,
        'eye_refine_strength':EYE_REFINE_STRENGTH if mode in JOINT_EYE_REFINE_MODES else None,
        'eye_refine_blend':EYE_REFINE_BLEND if mode in JOINT_EYE_REFINE_MODES else None,
        'extra_eye_passes_per_request':1 if mode in JOINT_EYE_REFINE_MODES and EYE_REFINE_BLEND>0 else 0,
        'eye_source':'HairFastGAN A' if mode in JOINT_EYE_MODES else None,
        'eye_margin_px_at_512':JOINT_EYE_MARGIN_PX,
        'eye_feather_px_at_512':JOINT_EYE_FEATHER_PX,
        'passes_per_prompt':1, 'face_control':'soft_blend' if mode in JOINT_CONTROLLED_MODES else 'none',
        'effective_face_blend':blend if mode in JOINT_CONTROLLED_MODES else 1.0,
        'face_blend_applied':mode in JOINT_CONTROLLED_MODES,
        'transition_px_at_512':JOINT_TRANSITION_PX, 'margin_px_at_512':JOINT_MARGIN_PX,
        'prompt_prefix':JOINT_INSTRUCTION,
        'mask_policy':'hair+face union, close small gaps, fill holes, expand outer perimeter',
        'outside_policy':'restore outside joint region with outer feather',
        'identity_conditioning':False, 'identity_guaranteed':False,
        'reference':'HairFastGAN output; no original-face identity conditioning',
    }}


In [ ]:
(OUT/'original_helpers.py').write_text('# P1–P5 เน้นคนละคุณสมบัติ; P6 รวมแบบสมดุล ไม่ต่อข้อความยาวทั้งหมด\nPROMPT_SET_VERSION = \'preserve_input_color_v3\'\nBASE_PROMPT = \'Photorealistic portrait. Preserve the input hair hue, saturation, brightness, highlights and shadows. Keep the same face, haircut and background. \'\nCOMMON_NEGATIVE = \'gray haze, washed-out hair, color shift, overexposure, plastic hair, wig, halo, changed haircut, deformed face\'\nFOCUS_PHRASES = {\n    \'prompt1_texture\': \'Refine fine irregular strand texture within existing locks. Keep their layout and gloss.\',\n    \'prompt2_color\': \'Retain the existing hair color and natural tonal variation. No recoloring or added gray veil.\',\n    \'prompt3_sharpness\': \'Resolve fine individual hair fibers and narrow strand separation without sharpening halos.\',\n    \'prompt4_blend\': \'Refine only the hair boundary into fine tapered strands. Preserve interior texture.\',\n    \'prompt5_lighting\': \'Keep the existing light direction and highlight intensity. Refine local strand shading without global relighting.\',\n    \'prompt6_combined\': \'Refine fine strand texture, fiber separation and tapered hair edges. Preserve local shading and gloss.\'\n}\nREFINE_PROMPTS = [{\'tag\':tag,\'prompt\':BASE_PROMPT+phrase,\'negative_prompt\':COMMON_NEGATIVE}\n                  for tag,phrase in FOCUS_PHRASES.items()]\n# Reference เข้าขั้น HairFastGAN เท่านั้น; diffusion นี้เห็น A ไม่ได้เห็น reference โดยตรง\nDEFAULT_PROMPT_TAG = \'prompt6_combined\'\nINPAINT_STEPS = 20\nINPAINT_STRENGTH = 0.5  # จุดเริ่มทดลอง ไม่ใช่ค่าที่รับรองว่าดีที่สุด\nINPAINT_GUIDANCE = 7.5\nDEFAULT_SEED = 3407\nHAIRFAST_SEED = 3407\nFACE_MARGIN_PX = 3      # pixels ที่ resolution ของภาพ A\nBLUR_RADIUS = 2.0\n# Fixed value range, ไม่ normalize ด้วย min/max ต่อภาพเพราะจะทำให้สีเปลี่ยน\n# official HairFast output ใช้ภาพในช่วง [0,1]; ถ้า repo ที่ใช้ต่างไปให้ตรวจและเปลี่ยน\nOUTPUT_TENSOR_RANGE = \'zero_one\'\nPROMPTS = {p[\'tag\']: p for p in REFINE_PROMPTS}\n\n\ndef segment_labels(image):\n    inputs = seg_processor(images=image.convert(\'RGB\'), return_tensors=\'pt\').to(DEVICE)\n    with torch.inference_mode():\n        logits = seg_model(**inputs).logits\n        logits = torch.nn.functional.interpolate(logits, size=image.size[::-1], mode=\'bilinear\', align_corners=False)\n    return logits.argmax(1)[0].cpu().numpy().astype(np.uint8)\n\n\ndef build_masks(labels, margin_px=FACE_MARGIN_PX, blur_radius=BLUR_RADIUS):\n    hair = labels == HAIR_ID\n    face = labels == FACE_ID\n    if not hair.any():\n        raise ValueError(\'ไม่พบเส้นผมในภาพ กรุณาตรวจภาพหรือ segmentation\')\n    if not face.any():\n        raise ValueError(\'ไม่พบใบหน้า ไม่สามารถใช้โหมดปกป้องหน้าได้\')\n    face_img = Image.fromarray(face.astype(np.uint8) * 255)\n    protected = np.asarray(face_img.filter(ImageFilter.MaxFilter(2 * margin_px + 1))) > 0\n    hair_img = Image.fromarray(hair.astype(np.uint8) * 255)\n    soft = np.asarray(hair_img.filter(ImageFilter.GaussianBlur(blur_radius)), dtype=np.float32) / 255\n    # C แก้เฉพาะผมที่ segmentation เลือก ไม่แก้ background/face แม้ขอบ blur ล้ำ\n    allowed = hair & ~protected\n    alpha = soft * allowed\n    if not np.any(alpha > 0):\n        raise ValueError(\'พื้นที่ผมหลังกันหน้าว่าง ต้องตรวจ mask\')\n    return {\'hair\': hair, \'face\': face, \'protected\': protected,\n            \'alpha\': alpha.astype(np.float32), \'raw_mask\': Image.fromarray(np.rint(soft * 255).astype(np.uint8))}\n\n\ndef composite_protected(base, edited, alpha):\n    if base.size != edited.size or np.asarray(alpha).shape != (base.height, base.width):\n        raise ValueError(\'ภาพและ alpha ต้องมีขนาดตรงกัน\')\n    a = np.clip(np.asarray(alpha, dtype=np.float32), 0, 1)[..., None]\n    original = np.array(base.convert(\'RGB\'))\n    changed = np.asarray(edited.convert(\'RGB\'), dtype=np.float32)\n    output = np.rint(original * (1 - a) + changed * a).clip(0, 255).astype(np.uint8)\n    output[a[..., 0] == 0] = original[a[..., 0] == 0]\n    return Image.fromarray(output)\n\n\ndef mask_overlay(image, masks):\n    a = np.asarray(image.convert(\'RGB\')).copy().astype(float)\n    for mask, rgb in [(masks[\'alpha\'] > 0, (0, 255, 0)), (masks[\'protected\'], (255, 0, 0))]:\n        a[mask] = 0.55 * a[mask] + 0.45 * np.array(rgb)\n    return Image.fromarray(a.astype(np.uint8))\n\n\ndef padded(image, target=512, mask=False):\n    scale = target / max(image.size)\n    size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))\n    resized = image.resize(size, Image.Resampling.NEAREST if mask else Image.Resampling.LANCZOS)\n    canvas = Image.new(image.mode, (target, target), 0)\n    x, y = (target-size[0])//2, (target-size[1])//2\n    canvas.paste(resized, (x, y))\n    return canvas, (x, y, x+size[0], y+size[1])\n\n\ndef tensor_to_pil(value, value_range=\'zero_one\'):\n    if isinstance(value, Image.Image):\n        return value.convert(\'RGB\')\n    if torch.is_tensor(value):\n        t = value.detach().cpu()\n        if t.ndim == 4 and t.shape[0] == 1:\n            t = t[0]\n        if t.ndim != 3 or t.shape[0] != 3:\n            raise ValueError(f\'Unexpected tensor shape: {tuple(t.shape)}\')\n        a = t.permute(1, 2, 0).numpy()\n    else:\n        a = np.asarray(value)\n    if a.dtype == np.uint8:\n        return Image.fromarray(a).convert(\'RGB\')\n    if value_range == \'minus_one_one\':\n        a = (a + 1) / 2\n    if not np.isfinite(a).all() or a.min() < -0.02 or a.max() > 1.02:\n        raise ValueError(\'Tensor range ไม่ตรง config: ตรวจ OUTPUT_TENSOR_RANGE ห้าม normalize อัตโนมัติ\')\n    return Image.fromarray(np.rint(a.clip(0, 1) * 255).astype(np.uint8)).convert(\'RGB\')\n\n\ndef resolve_prompt_configs(overrides=None):\n    # คัดลอกต่อคำขอ ไม่เปลี่ยน preset ส่วนกลาง\n    configs = {tag:dict(cfg) for tag,cfg in PROMPTS.items()}\n    if overrides is None:\n        return configs\n    if not isinstance(overrides, dict) or len(overrides) > len(PROMPTS):\n        raise ValueError(\'prompt_overrides ต้องเป็น object ตามชื่อ prompt ที่รองรับ\')\n    for tag, change in overrides.items():\n        if tag not in configs or not isinstance(change, dict):\n            raise ValueError(\'ชื่อ prompt หรือรูปแบบ override ไม่ถูกต้อง\')\n        if set(change) - {\'prompt\',\'negative_prompt\'}:\n            raise ValueError(\'แก้ได้เฉพาะ prompt และ negative_prompt\')\n        for field, text in change.items():\n            if not isinstance(text, str) or len(text) > 4000:\n                raise ValueError(\'ข้อความ prompt ต้องเป็น string ไม่เกิน 4000 ตัวอักษร\')\n            if field == \'prompt\' and not text.strip():\n                raise ValueError(\'prompt ห้ามว่าง\')\n            configs[tag][field] = text\n    return configs\n\n\ndef parse_prompt_changes(prompt_tag, mode, prompt=None, negative_prompt=None, prompt_overrides=None):\n    import json\n    if prompt_tag not in PROMPTS:\n        raise ValueError(\'ไม่พบ prompt_tag\')\n    if prompt_overrides is not None:\n        if not isinstance(prompt_overrides, str) or len(prompt_overrides) > 60000:\n            raise ValueError(\'prompt_overrides ต้องเป็น JSON string ไม่เกิน 60000 ตัวอักษร\')\n        try:\n            changes = json.loads(prompt_overrides)\n        except (ValueError, TypeError):\n            raise ValueError(\'prompt_overrides เป็น JSON ไม่ถูกต้อง\')\n        resolve_prompt_configs(changes)\n        if not isinstance(changes, dict):\n            raise ValueError(\'prompt_overrides ต้องเป็น JSON object\')\n    else:\n        changes = {}\n    if prompt is not None or negative_prompt is not None:\n        if mode in (\'all_prompts\', \'all_prompts_unprotected\', \'all_prompts_joint_controlled\',\'all_prompts_joint_unprotected\',\'all_prompts_joint_controlled_noeyes\',\'all_prompts_joint_unprotected_noeyes\',\'all_prompts_joint_controlled_eyerefine\',\'all_prompts_joint_unprotected_eyerefine\'):\n            raise ValueError(\'all_prompts ให้ใช้ prompt_overrides เพื่อระบุข้อความแยกแต่ละชุด\')\n        if prompt_tag in changes:\n            raise ValueError(\'ส่งข้อความซ้ำกัน: เลือกใช้ prompt/negative_prompt หรือ prompt_overrides\')\n        changes[prompt_tag] = {}\n        if prompt is not None:\n            changes[prompt_tag][\'prompt\'] = prompt\n        if negative_prompt is not None:\n            changes[prompt_tag][\'negative_prompt\'] = negative_prompt\n    if mode == \'baseline\' and changes:\n        raise ValueError(\'baseline ไม่ใช้ diffusion prompt ให้เลือก protected/abc/all_prompts\')\n    if mode not in (\'all_prompts\', \'all_prompts_unprotected\', \'all_prompts_joint_controlled\',\'all_prompts_joint_unprotected\',\'all_prompts_joint_controlled_noeyes\',\'all_prompts_joint_unprotected_noeyes\',\'all_prompts_joint_controlled_eyerefine\',\'all_prompts_joint_unprotected_eyerefine\') and set(changes) - {prompt_tag}:\n        raise ValueError(\'โหมดนี้แก้ได้เฉพาะ prompt_tag ที่เลือก\')\n    resolve_prompt_configs(changes)\n    return changes\n\n\ndef resolve_inpaint_settings(strength=None, steps=None, guidance=None):\n    import math\n    strength = float(INPAINT_STRENGTH if strength is None else strength)\n    steps = INPAINT_STEPS if steps is None else steps\n    guidance = float(INPAINT_GUIDANCE if guidance is None else guidance)\n    if isinstance(steps, bool) or not isinstance(steps, (int, np.integer)) or not 1 <= steps <= 100:\n        raise ValueError(\'steps ต้องเป็นจำนวนเต็ม 1–100\')\n    steps = int(steps)\n    if not math.isfinite(strength) or not 0 < strength <= 1:\n        raise ValueError(\'strength ต้องมากกว่า 0 และไม่เกิน 1\')\n    if not math.isfinite(guidance) or not 1 <= guidance <= 20:\n        raise ValueError(\'guidance ต้องอยู่ในช่วง 1–20\')\n    if int(steps * strength) < 1:\n        raise ValueError(\'steps × strength ต้องไม่น้อยกว่า 1\')\n    return {\'strength\':strength, \'steps\':steps, \'guidance\':guidance}\n\n\ndef refine_once(\n    base, masks, prompt_config, seed, protected,\n    strength=None, steps=None, guidance=None, trace_dir=None\n):\n    validate_joint_prompt_lengths({prompt_config.get(\'tag\',\'custom\'):prompt_config})\n    settings = resolve_inpaint_settings(strength, steps, guidance)\n    image512, box = padded(base)\n\n    if protected:\n        alpha_img = Image.fromarray(\n            np.rint(masks[\'alpha\'] * 255).astype(np.uint8)\n        )\n        mask512, _ = padded(alpha_img, mask=True)\n\n        protect512, _ = padded(\n            Image.fromarray(\n                masks[\'protected\'].astype(np.uint8) * 255\n            ),\n            mask=True\n        )\n        protect512 = np.asarray(\n            protect512.filter(ImageFilter.MaxFilter(3))\n        ) > 0\n\n        m = np.array(mask512)\n        m[protect512] = 0\n        mask512 = Image.fromarray(m)\n    else:\n        mask512, _ = padded(masks[\'raw_mask\'], mask=True)\n\n    if not (np.asarray(mask512) > 127).any():\n        raise ValueError(\'mask เล็กเกินไปหลัง resize เป็น 512\')\n\n    # ===== เพิ่ม: ตรวจภาพและ mask ก่อนเข้า Diffusion =====\n    from IPython.display import display\n\n    print("\\nPrompt:", prompt_config.get(\'tag\', \'custom\'))\n    print("ข้อความที่ส่งจริง:", prompt_config[\'prompt\'])\n    print("Negative:", prompt_config[\'negative_prompt\'])\n    print("ค่าที่ใช้:", settings, "| Seed:", seed)\n    print("ปกป้องหน้าใน refine_once:", protected)\n    # Joint ส่ง protected=False ตรงนี้ แล้วผสมภาพใน refine_joint ภายหลัง\n\n    image_debug = image512.convert("RGB")\n    mask_debug = mask512.convert("L")\n\n    # สีแดงทับบริเวณ mask ที่มีค่ามากกว่า 127\n    selected = np.asarray(mask_debug) > 127\n    overlay = np.asarray(image_debug).copy()\n    overlay[selected] = (\n        overlay[selected].astype(np.float32) * 0.55\n        + np.array([255, 0, 0], dtype=np.float32) * 0.45\n    ).astype(np.uint8)\n\n    width, height = image_debug.size\n    panel = Image.new("RGB", (width * 3, height))\n    panel.paste(image_debug, (0, 0))\n    panel.paste(mask_debug.convert("RGB"), (width, 0))\n    panel.paste(Image.fromarray(overlay), (width * 2, 0))\n\n    print("ซ้าย: ภาพเข้าโมเดล | กลาง: mask | ขวา: พื้นที่แก้สีแดง")\n    display(panel)\n    # ===== จบส่วนตรวจ =====\n\n    generator = torch.Generator(\n        device=DEVICE\n    ).manual_seed(int(seed))\n\n    with torch.inference_mode():\n        result = inpaint_pipe(\n            image=image512,\n            mask_image=mask512,\n            prompt=prompt_config[\'prompt\'],\n            negative_prompt=prompt_config[\'negative_prompt\'],\n            num_inference_steps=settings["steps"],\n            strength=settings["strength"],\n            guidance_scale=settings["guidance"],\n            generator=generator\n        )\n\n    flags = getattr(result, \'nsfw_content_detected\', None)\n    if flags is not None and any(flags):\n        raise ValueError(\'ผลภาพถูกตัวกรองของโมเดลระงับ\')\n\n    edited = (\n        result.images[0]\n        .convert(\'RGB\')\n        .crop(box)\n        .resize(base.size, Image.Resampling.LANCZOS)\n    )\n\n    if trace_dir is not None:\n        trace_dir = Path(trace_dir)\n        trace_dir.mkdir(parents=True, exist_ok=True)\n        image512.save(trace_dir/\'diffusion_input_512.png\')\n        mask512.save(trace_dir/\'diffusion_mask_512.png\')\n        result.images[0].convert(\'RGB\').save(trace_dir/\'diffusion_raw_512.png\')\n        edited.save(trace_dir/\'diffusion_raw_before_blend.png\')\n        # Resize-only control isolates resolution loss from the generated changes.\n        image512.crop(box).resize(base.size, Image.Resampling.LANCZOS).save(trace_dir/\'resize_only_control.png\')\n\n    return (\n        composite_protected(base, edited, masks[\'alpha\'])\n        if protected\n        else edited\n    )\n\n\n# Single-pass joint hair + face study: same six prompts/noise/mask in both modes.\nJOINT_MODES = (\'all_prompts_joint_controlled\',\'all_prompts_joint_unprotected\',\'all_prompts_joint_controlled_noeyes\',\'all_prompts_joint_unprotected_noeyes\',\'all_prompts_joint_controlled_eyerefine\',\'all_prompts_joint_unprotected_eyerefine\')\nJOINT_FACE_BLEND = 0.35\nJOINT_TRANSITION_PX = 24  # measured at 512px short side; scaled with image size\nJOINT_MARGIN_PX = 12\nJOINT_INSTRUCTION = BASE_PROMPT\nJOINT_FOCUS_PHRASES = dict(FOCUS_PHRASES)\n\nJOINT_CONTROLLED_MODES = (\'all_prompts_joint_controlled\', \'all_prompts_joint_controlled_noeyes\', \'all_prompts_joint_controlled_eyerefine\')\nJOINT_EYE_MODES = (\'all_prompts_joint_controlled\', \'all_prompts_joint_unprotected\',\'all_prompts_joint_controlled_eyerefine\',\'all_prompts_joint_unprotected_eyerefine\')\n\ndef resolve_joint_settings(face_blend=JOINT_FACE_BLEND):\n    import math\n    face_blend = float(face_blend)\n    if not math.isfinite(face_blend) or not 0 <= face_blend <= 1:\n        raise ValueError(\'face_blend ต้องอยู่ในช่วง 0–1\')\n    return {\'face_blend\':face_blend}\n\n\ndef joint_prompt_configs(configs):\n    # Keep P1–P6 axes; replace the hair-only preamble with a short joint preamble.\n    # Compact P6 to avoid swallowing its five axes in a long prompt.\n    result = {}\n    for tag,cfg in configs.items():\n        body = JOINT_FOCUS_PHRASES[tag] if cfg[\'prompt\'] == PROMPTS[tag][\'prompt\'] else cfg[\'prompt\']\n        result[tag] = {**cfg, \'prompt\':JOINT_INSTRUCTION + body}\n    return result\n\n\ndef validate_joint_prompt_lengths(configs):\n    # Validate using the tokenizer actually loaded in Colab, not character counts.\n    tokenizer = getattr(inpaint_pipe, \'tokenizer\', None)\n    if tokenizer is None:\n        return\n    limit = tokenizer.model_max_length\n    for tag,cfg in configs.items():\n        for field in (\'prompt\',\'negative_prompt\'):\n            count = len(tokenizer(cfg[field], truncation=False)[\'input_ids\'])\n            if count > limit:\n                raise ValueError(f\'{tag} {field}: {count} tokens เกิน {limit}; ย่อข้อความเพื่อไม่ให้ส่วนท้ายถูกตัด\')\n\n\n# รักษาตาจากภาพ A ใน Joint ทั้งสองโหมด (ปิดได้เพื่อเทียบผลเดิม)\n# Eye protection เลือกตาม mode ของแต่ละคำขอ ไม่ใช้ global switch\nJOINT_EYE_MARGIN_PX = 3     # ที่ด้านสั้น 512: ครอบขอบเปลือกตา\nJOINT_EYE_FEATHER_PX = 6    # ไล่ขอบออกด้านนอก ไม่ลดการปกป้องแกนกลาง\n_EYE_LANDMARK_MODEL = None\n\n\ndef detect_eye_landmarks(base):\n    """ใช้ face-alignment ที่ติดตั้งไว้แล้ว; โหลดครั้งแรก ใช้ CPU ลด VRAM."""\n    global _EYE_LANDMARK_MODEL\n    import face_alignment\n    import inspect\n    if _EYE_LANDMARK_MODEL is None:\n        kwargs = dict(device=\'cpu\', flip_input=False)\n        if \'compile\' in inspect.signature(face_alignment.FaceAlignment).parameters:\n            kwargs[\'compile\'] = False\n        _EYE_LANDMARK_MODEL = face_alignment.FaceAlignment(\n            face_alignment.LandmarksType.TWO_D, **kwargs)\n    with torch.inference_mode():\n        predictions = _EYE_LANDMARK_MODEL.get_landmarks(np.asarray(base.convert(\'RGB\')))\n    if predictions is None or len(predictions) != 1:\n        raise ValueError(\'Eye protection: ต้องตรวจพบใบหน้าเดียวในภาพ A กรุณาใช้ภาพหน้าชัดเจน\')\n    points = np.asarray(predictions[0], dtype=np.float32)\n    if points.shape != (68, 2) or not np.isfinite(points).all():\n        raise ValueError(\'Eye protection: landmarks ไม่ถูกต้อง\')\n    return points\n\n\ndef build_eye_protection(size, landmarks):\n    """eye_keep=1 คืน A เต็มที่; 0 ใช้ Joint ตามเดิม; ขอบมีช่วงเปลี่ยนที่จำกัด."""\n    from scipy.ndimage import distance_transform_edt\n    w, h = size\n    canvas = Image.new(\'L\', (w, h), 0)\n    draw = ImageDraw.Draw(canvas)\n    for eye in (landmarks[36:42], landmarks[42:48]):\n        if np.ptp(eye[:, 0]) < 2 or (eye < 0).any() or (eye[:, 0] >= w).any() or (eye[:, 1] >= h).any():\n            raise ValueError(\'Eye protection: ตำแหน่งตาอยู่นอกภาพหรือเล็กเกินไป\')\n        draw.polygon([tuple(map(float, p)) for p in eye], fill=255)\n    scale = min(w, h) / 512.0\n    margin = max(0, round(JOINT_EYE_MARGIN_PX * scale))\n    if margin:\n        canvas = canvas.filter(ImageFilter.MaxFilter(2 * margin + 1))\n    core = np.asarray(canvas) > 0\n    feather = max(1.0, JOINT_EYE_FEATHER_PX * scale)\n    x = np.clip(distance_transform_edt(~core) / feather, 0, 1)\n    keep = (1 - x*x*(3-2*x)).astype(np.float32)\n    keep[core] = 1.0\n    return {\'eye_core\': core, \'eye_keep\': keep}\n\n\ndef build_joint_masks(masks, face_blend=JOINT_FACE_BLEND, base=None, mode=\'all_prompts_joint_controlled\'):\n    if mode not in JOINT_MODES:\n        raise ValueError(\'ไม่ใช่โหมด joint\')\n    from scipy.ndimage import binary_fill_holes, distance_transform_edt\n    face_blend = resolve_joint_settings(face_blend)[\'face_blend\']\n    h,w = masks[\'face\'].shape\n    scale = min(h,w) / 512.0\n    margin = max(1, round(JOINT_MARGIN_PX * scale))\n    transition = max(1.0, JOINT_TRANSITION_PX * scale)\n    # Fill enclosed holes (e.g. glasses) and close small segmentation gaps at hairline.\n    union = masks[\'hair\'] | masks[\'face\']\n    radius = max(1, round(3 * scale))\n    u = Image.fromarray(union.astype(np.uint8)*255)\n    closed = np.asarray(u.filter(ImageFilter.MaxFilter(2*radius+1)).filter(\n        ImageFilter.MinFilter(2*radius+1))) > 0\n    core = binary_fill_holes(union | closed)\n    core_img = Image.fromarray(core.astype(np.uint8)*255)\n    support = np.asarray(core_img.filter(ImageFilter.MaxFilter(2*margin+1))) > 0\n    # Feather only the outer perimeter; never cut the shared face/hair boundary.\n    distance = distance_transform_edt(np.pad(support, 1))[1:-1,1:-1]\n    x = np.clip(distance / margin, 0, 1)\n    outer_alpha = (x*x*(3-2*x)).astype(np.float32)\n    outer_alpha[core] = 1.0\n    # Smoothly reduce generated-image weight towards the interior of the face.\n    face = binary_fill_holes(masks[\'face\']) & ~masks[\'hair\']\n    distance_face = distance_transform_edt(np.pad(face, 1))[1:-1,1:-1]\n    x = np.clip(distance_face / transition, 0, 1)\n    face_weight = (x*x*(3-2*x)).astype(np.float32)\n    controlled_alpha = outer_alpha * (1 - (1-face_blend)*face_weight)\n    if mode in JOINT_EYE_MODES:\n        if base is not None:\n            if base.size != (w, h):\n                raise ValueError(\'Eye protection: ภาพ A และ mask ขนาดไม่ตรงกัน\')\n            masks.update(build_eye_protection(base.size, detect_eye_landmarks(base)))\n        if \'eye_keep\' not in masks:\n            raise ValueError(\'Eye protection: ต้องส่งภาพ A เพื่อหาตำแหน่งตาก่อน\')\n        eye_keep, eye_core = masks[\'eye_keep\'], masks[\'eye_core\']\n    else:\n        eye_keep = np.zeros((h, w), dtype=np.float32)\n        eye_core = np.zeros((h, w), dtype=bool)\n    return {\'eye_keep\':eye_keep, \'eye_core\':eye_core,\n            \'raw_mask\':Image.fromarray(support.astype(np.uint8)*255),\n            \'core\':core, \'support\':support, \'face_weight\':face_weight,\n            \'outer_alpha\':outer_alpha, \'controlled_alpha\':controlled_alpha,\n            \'margin_px_actual\':margin, \'transition_px_actual\':transition}\n\n\ndef joint_alpha(joint, mode):\n    if mode not in JOINT_MODES:\n        raise ValueError(\'ไม่ใช่โหมด joint\')\n    alpha = joint[\'controlled_alpha\'] if mode in JOINT_CONTROLLED_MODES else joint[\'outer_alpha\']\n    return alpha * (1 - joint[\'eye_keep\']) if mode in JOINT_EYE_MODES else alpha.copy()\n\n\n\nJOINT_EYE_REFINE_MODES = (\'all_prompts_joint_controlled_eyerefine\',\'all_prompts_joint_unprotected_eyerefine\')\nEYE_REFINE_STRENGTH = 0.20  # ความแรงการสร้างรายละเอียดตา แยกจาก Strength ของ Joint\nEYE_REFINE_BLEND = 0.35     # น้ำหนักตาที่ AI ปรับ; ที่เหลือใช้ A\nEYE_REFINE_STEPS = 30\nEYE_REFINE_GUIDANCE = 4.0\nEYE_REFINE_PROMPT = (\'Photorealistic eyes, subtly refine existing eye detail. Preserve eye shape, \'\n    \'iris size, iris color, gaze direction and eyelids. Natural texture and soft lighting.\')\nEYE_REFINE_NEGATIVE = (\'different eye shape, different iris color, changed gaze, enlarged eyes, \'\n    \'asymmetric pupils, extra pupils, heavy makeup, oversharpening, artifacts\')\n\n\ndef refine_eyes_from_a(base, joint, seed):\n    # Cache ต่อคำขอ: ทั้ง P1–P6 ใช้ตาปรับชุดเดียวกัน ไม่สร้างหกรอบ\n    import hashlib\n    key=(hashlib.sha256(base.tobytes()).hexdigest(), int(seed), EYE_REFINE_STRENGTH,\n         EYE_REFINE_BLEND,EYE_REFINE_STEPS,EYE_REFINE_GUIDANCE,\n         EYE_REFINE_PROMPT,EYE_REFINE_NEGATIVE)\n    if joint.get(\'_eye_cache_key\')==key:\n        return joint[\'_eye_cache_image\']\n    if not 0<=EYE_REFINE_BLEND<=1:\n        raise ValueError(\'EYE_REFINE_BLEND ต้องอยู่ในช่วง 0–1\')\n    if EYE_REFINE_BLEND==0:\n        return base.copy()\n    ys,xs=np.where(joint[\'eye_keep\']>0)\n    if len(xs)==0: raise ValueError(\'ไม่พบพื้นที่ตาสำหรับปรับรายละเอียด\')\n    # ครอปตาคู่พร้อมบริบทจาก A เพื่อไม่ย่อทั้งหน้าแล้วสูญรายละเอียดตา\n    width=xs.max()-xs.min()+1; height=ys.max()-ys.min()+1\n    padx=max(16,round(width*.25));pady=max(16,round(height*1.5))\n    box=(max(0,int(xs.min()-padx)),max(0,int(ys.min()-pady)),\n         min(base.width,int(xs.max()+1+padx)),min(base.height,int(ys.max()+1+pady)))\n    x0,y0,x1,y1=box\n    crop=base.crop(box)\n    raw_mask=Image.fromarray((joint[\'eye_keep\'][y0:y1,x0:x1]>0).astype(np.uint8)*255)\n    cfg={\'tag\':\'eye_detail_from_A\',\'prompt\':EYE_REFINE_PROMPT,\'negative_prompt\':EYE_REFINE_NEGATIVE}\n    validate_joint_prompt_lengths({\'eye_detail_from_A\':cfg})\n    edited=refine_once(crop,{\'raw_mask\':raw_mask},cfg,seed,False,\n        strength=EYE_REFINE_STRENGTH,steps=EYE_REFINE_STEPS,guidance=EYE_REFINE_GUIDANCE)\n    canvas=base.copy();canvas.paste(edited,box[:2])\n    # น้ำหนักคงที่ก่อนประกอบด้วย eye_keep อีกครั้ง: ป้องกัน feather ซ้ำ\n    candidate=composite_protected(base,canvas,np.full((base.height,base.width),EYE_REFINE_BLEND,dtype=np.float32))\n    joint[\'_eye_cache_key\']=key;joint[\'_eye_cache_image\']=candidate\n    return candidate\n\ndef refine_joint(base, masks, joint, prompt_config, seed, mode, **settings):\n    # Exactly one diffusion call; both modes generate with the same full joint mask.\n    inference_masks = {**masks, \'raw_mask\':joint[\'raw_mask\']}\n    generated = refine_once(base, inference_masks, prompt_config, seed, False, **settings)\n    # Eye modes คืนตาจาก A; noeyes ไม่มีการตัดตาออกจากน้ำหนัก Blend\n    result = composite_protected(base, generated, joint_alpha(joint, mode))\n    if mode in JOINT_EYE_REFINE_MODES:\n        eye_image = refine_eyes_from_a(base, joint, seed)\n        # เริ่มจากน้ำหนัก Joint ปกติ เพื่อไม่ feather ตา A ซ้ำ\n        normal_alpha = joint[\'controlled_alpha\'] if mode in JOINT_CONTROLLED_MODES else joint[\'outer_alpha\']\n        result = composite_protected(base, generated, normal_alpha)\n        result = composite_protected(result, eye_image, joint[\'eye_keep\'])\n    return result\n\n\ndef joint_metadata(mode, face_blend):\n    if mode not in JOINT_MODES:\n        return {}\n    blend = resolve_joint_settings(face_blend)[\'face_blend\']\n    return {\'joint_refinement\': {\n        \'eye_protection\':mode in JOINT_EYE_MODES and mode not in JOINT_EYE_REFINE_MODES,\n        \'eye_refinement\':mode in JOINT_EYE_REFINE_MODES,\n        \'eye_refine_strength\':EYE_REFINE_STRENGTH if mode in JOINT_EYE_REFINE_MODES else None,\n        \'eye_refine_blend\':EYE_REFINE_BLEND if mode in JOINT_EYE_REFINE_MODES else None,\n        \'extra_eye_passes_per_request\':1 if mode in JOINT_EYE_REFINE_MODES and EYE_REFINE_BLEND>0 else 0,\n        \'eye_source\':\'HairFastGAN A\' if mode in JOINT_EYE_MODES else None,\n        \'eye_margin_px_at_512\':JOINT_EYE_MARGIN_PX,\n        \'eye_feather_px_at_512\':JOINT_EYE_FEATHER_PX,\n        \'passes_per_prompt\':1, \'face_control\':\'soft_blend\' if mode in JOINT_CONTROLLED_MODES else \'none\',\n        \'effective_face_blend\':blend if mode in JOINT_CONTROLLED_MODES else 1.0,\n        \'face_blend_applied\':mode in JOINT_CONTROLLED_MODES,\n        \'transition_px_at_512\':JOINT_TRANSITION_PX, \'margin_px_at_512\':JOINT_MARGIN_PX,\n        \'prompt_prefix\':JOINT_INSTRUCTION,\n        \'mask_policy\':\'hair+face union, close small gaps, fill holes, expand outer perimeter\',\n        \'outside_policy\':\'restore outside joint region with outer feather\',\n        \'identity_conditioning\':False, \'identity_guaranteed\':False,\n        \'reference\':\'HairFastGAN output; no original-face identity conditioning\',\n    }}\n',encoding='utf-8')
assert sha(OUT/'original_helpers.py')==ORIGINAL_HELPER_SHA256


## ตรวจ mask ตาอัตโนมัติและลองซ้ำเฉพาะภาพที่ไม่ผ่าน

In [ ]:
from transformers import SegformerImageProcessor,SegformerForSemanticSegmentation
from huggingface_hub import HfApi
DEVICE='cuda'
for module in [enc,pp,G,align,hair_fast.embed.encoder,hair_fast.embed.e4e,id_net,parser_net]:module.cpu()
gc.collect();torch.cuda.empty_cache()
SEG_MODEL_ID='mattmdjaga/segformer_b2_clothes'
if 'seg_model' not in globals():
    SEG_REVISION=HfApi().model_info(SEG_MODEL_ID).sha
    seg_processor=SegformerImageProcessor.from_pretrained(SEG_MODEL_ID,revision=SEG_REVISION)
    seg_model=SegformerForSemanticSegmentation.from_pretrained(SEG_MODEL_ID,revision=SEG_REVISION)
seg_model.to(DEVICE).eval()
label_ids={str(v).lower():int(k) for k,v in seg_model.config.id2label.items()}
HAIR_ID,FACE_ID=label_ids['hair'],label_ids['face'];assert (HAIR_ID,FACE_ID)==(2,11)
assert MODE in ['all_prompts_joint_controlled','all_prompts_joint_unprotected']
assert PROMPT_TAG in PROMPTS and PROMPT_TAG=='prompt6_combined'
assert (INPAINT_STRENGTH,INPAINT_STEPS,INPAINT_GUIDANCE,JOINT_FACE_BLEND)==(.5,20,7.5,.35)
PROMPT_CONFIG=joint_prompt_configs({PROMPT_TAG:PROMPTS[PROMPT_TAG]})[PROMPT_TAG]
settings=resolve_inpaint_settings(CONFIG['strength'],CONFIG['steps'],CONFIG['guidance'])
face_blend=resolve_joint_settings(CONFIG['face_blend'])['face_blend']
(OUT/'actual_prompt.json').write_text(json.dumps(PROMPT_CONFIG,indent=2))
print('Original prompt:',PROMPT_CONFIG['prompt']);print('Negative:',PROMPT_CONFIG['negative_prompt'])
MANUAL_ROOT=ROOT/'reviewed_eye_annotations';MANUAL_ROOT.mkdir(exist_ok=True)
MASK_ROOT=OUT/'mask_cache';MASK_ROOT.mkdir(exist_ok=True)

def atomic_json(path,data):
    tmp=Path(str(path)+'.tmp');tmp.write_text(json.dumps(data,ensure_ascii=False,indent=2));os.replace(tmp,path)

# Observe the original face-alignment calls without changing their results.
import inspect
import importlib.metadata as _metadata
import face_alignment
EYE_AUDIT_ROOT=OUT/'eye_debug'
EYE_AUDIT_ROOT.mkdir(exist_ok=True)
EYE_CALL_CONTEXT={}

class EyeDetectorAudit:
    def __init__(self,model):self.model=model
    def __getattr__(self,name):return getattr(self.model,name)
    def get_landmarks(self,image,*args,**kwargs):
        array=np.asarray(image)
        digest=hashlib.sha256(array.tobytes()).hexdigest()
        row=dict(EYE_CALL_CONTEXT,input_rgb_sha256=digest,shape=list(array.shape),
                 dtype=str(array.dtype),minimum=int(array.min()),maximum=int(array.max()))
        try:
            result=self.model.get_landmarks(image,*args,**kwargs)
            row['faces_detected']=0 if result is None else len(result)
            row['landmark_shapes']=[] if result is None else [list(np.asarray(p).shape) for p in result]
            row['status']='one_face' if row['faces_detected']==1 else 'no_face' if row['faces_detected']==0 else 'multiple_faces'
            if row['faces_detected']!=1:
                Image.fromarray(array).save(EYE_AUDIT_ROOT/(digest+'.png'))
            return result
        except Exception as error:
            row.update(status='detector_exception',error_type=type(error).__name__,message=str(error))
            raise
        finally:
            with (EYE_AUDIT_ROOT/'detector_calls.jsonl').open('a') as stream:
                stream.write(json.dumps(row,ensure_ascii=False)+'\n')

# These constructor arguments match detect_eye_landmarks in the supplied notebook.
_eye_kwargs=dict(device='cpu',flip_input=False)
if 'compile' in inspect.signature(face_alignment.FaceAlignment).parameters:_eye_kwargs['compile']=False
if _EYE_LANDMARK_MODEL is None:
    _EYE_LANDMARK_MODEL=face_alignment.FaceAlignment(face_alignment.LandmarksType.TWO_D,**_eye_kwargs)
if not isinstance(_EYE_LANDMARK_MODEL,EyeDetectorAudit):
    # Avoid nesting proxies when this cell is rerun.
    if type(_EYE_LANDMARK_MODEL).__name__=='EyeDetectorAudit':_EYE_LANDMARK_MODEL=_EYE_LANDMARK_MODEL.model
    _EYE_LANDMARK_MODEL=EyeDetectorAudit(_EYE_LANDMARK_MODEL)
_eye_versions={}
for package in ['face-alignment','torch','numpy','Pillow','transformers','diffusers']:
    try:_eye_versions[package]=_metadata.version(package)
    except _metadata.PackageNotFoundError:_eye_versions[package]='not installed'
atomic_json(EYE_AUDIT_ROOT/'environment.json',dict(packages=_eye_versions,constructor=_eye_kwargs,
    landmark_type='TWO_D',mode=MODE,helper_sha256=ORIGINAL_HELPER_SHA256,
    note='Logs observe original detector calls; no face selection, threshold change or fabricated landmarks.'))

def export_eye_debug():
    log=EYE_AUDIT_ROOT/'detector_calls.jsonl'
    if log.is_file():
        rows=[json.loads(line) for line in log.read_text().splitlines() if line.strip()]
        pd.DataFrame(rows).to_csv(EYE_AUDIT_ROOT/'detector_calls.csv',index=False)
    atomic_json(EYE_AUDIT_ROOT/'pending_eye_masks.json',PENDING_MASKS)
    pending={r['sha256']:r for r in PENDING_MASKS}
    if pending:
        tiles=[]
        for digest,r in pending.items():
            with Image.open(r['path']) as im:
                base=im.convert('RGB');base.save(EYE_AUDIT_ROOT/(digest+'_original.png'))
                tile=Image.new('RGB',(360,400),'white')
                thumb=ImageOps.contain(base,(360,360));tile.paste(thumb,((360-thumb.width)//2,0))
            ImageDraw.Draw(tile).text((5,365),f"{r['split']} {r['trial']} {r['case']}\n{r['branch']}",fill='black')
            tiles.append(tile)
        panel=Image.new('RGB',(360*3,400*((len(tiles)+2)//3)),'white')
        for i,tile in enumerate(tiles):panel.paste(tile,((i%3)*360,(i//3)*400))
        panel.save(EYE_AUDIT_ROOT/'pending_contact_sheet.jpg')
        from IPython.display import display
        display(panel)
    archive=shutil.make_archive(str(OUT/'eye_debug'),'zip',EYE_AUDIT_ROOT)
    print('Eye diagnostic archive:',archive)
    return archive


def mask_paths(rec):
    stem=rec['sha256'];return MASK_ROOT/(stem+'.npz'),MASK_ROOT/(stem+'.json')

def mask_spec(rec):
    annotation=MANUAL_ROOT/(rec['sha256']+'.json')
    return dict(image_sha256=rec['sha256'],helper_sha256=ORIGINAL_HELPER_SHA256,mode=MODE,face_blend=face_blend,
                seg_revision=getattr(seg_model.config,'_commit_hash',None),annotation_sha256=sha(annotation) if annotation.is_file() else None)

# Batch-only recovery: original detector, unchanged diffusion image and helper functions.
EYE_RECOVERY_POLICY='same_detector_resize_pad_v1'

def recover_eye_landmarks(base, face_mask):
    """Retry only failed original detections. Never reuse another branch's eyes."""
    w,h=base.size
    attempts=[]
    for side,pad_fraction in [(512,0.0),(768,0.0),(512,0.20),(768,0.20)]:
        scale=side/max(w,h)
        rw,rh=max(1,round(w*scale)),max(1,round(h*scale))
        pad=round(max(rw,rh)*pad_fraction)
        resized=base.resize((rw,rh),Image.Resampling.LANCZOS)
        # Edge padding, with a known inverse transform; no crop or generated pixels.
        arr=np.pad(np.asarray(resized),((pad,pad),(pad,pad),(0,0)),mode='edge')
        detail=dict(long_side=side,pad=pad,resized_size=[rw,rh])
        EYE_CALL_CONTEXT['stage']=f'retry_{side}_pad_{pad}'
        try:
            points=detect_eye_landmarks(Image.fromarray(arr)).copy()
            points[:,0]=(points[:,0]-pad+.5)*w/rw-.5
            points[:,1]=(points[:,1]-pad+.5)*h/rh-.5
            protection=build_eye_protection(base.size,points)
            # Validate recovered eye centers against this image's segmented face.
            centers=np.stack([points[36:42].mean(0),points[42:48].mean(0)])
            if not ((centers[:,0]>=0)&(centers[:,0]<w)&(centers[:,1]>=0)&(centers[:,1]<h)).all():
                raise ValueError('Eye protection: recovered eye centers outside image')
            xy=np.rint(centers).astype(int);xy[:,0]=np.clip(xy[:,0],0,w-1);xy[:,1]=np.clip(xy[:,1],0,h-1)
            if not face_mask[xy[:,1],xy[:,0]].all():
                raise ValueError('Eye protection: recovered eye centers outside segmented face')
            separation=float(np.linalg.norm(centers[0]-centers[1]))
            if not .025*min(w,h)<=separation<=.65*max(w,h):
                raise ValueError('Eye protection: implausible recovered eye separation')
            detail['status']='accepted';attempts.append(detail)
            return protection,attempts
        except ValueError as error:
            if not str(error).startswith('Eye protection:'):raise
            detail.update(status='failed',reason=str(error));attempts.append(detail)
    error=ValueError('Eye protection: original and resize/pad retries failed; inspect pending mask preview')
    error.eye_attempts=attempts
    raise error


def create_mask(rec):
    path,meta=mask_paths(rec);spec=mask_spec(rec)
    if path.is_file() and meta.is_file() and json.loads(meta.read_text())['spec']==spec:return
    EYE_CALL_CONTEXT.clear();EYE_CALL_CONTEXT.update({k:rec[k] for k in ['split','trial','case','branch']})
    EYE_CALL_CONTEXT['stage']='original'
    with Image.open(rec['path']) as im:base=im.convert('RGB').copy()
    masks=build_masks(segment_labels(base))
    attempts=[]
    ann=MANUAL_ROOT/(rec['sha256']+'.json')
    if ann.is_file():
        a=json.loads(ann.read_text());assert a['image_sha256']==rec['sha256'] and a['size']==list(base.size) and a['reviewed'] is True
        masks.update(build_eye_protection(base.size,np.asarray(a['landmarks'],np.float32)))
        joint=build_joint_masks(masks,face_blend,base=None,mode=MODE)
        source='manual_reviewed'
    else:
        try:
            joint=build_joint_masks(masks,face_blend,base=base,mode=MODE)
            source='original_auto_landmarks'
        except ValueError as error:
            if not str(error).startswith('Eye protection:'):raise
            protection,attempts=recover_eye_landmarks(base,masks['face'])
            masks.update(protection)
            joint=build_joint_masks(masks,face_blend,base=None,mode=MODE)
            source=EYE_RECOVERY_POLICY
    assert joint['eye_core'].any() and np.all(joint_alpha(joint,MODE)[joint['eye_core']]==0)
    values={}
    for prefix,items in [('m',masks),('j',joint)]:
        for k,v in items.items():values[prefix+'__'+k]=np.asarray(v) if isinstance(v,Image.Image) else v
    np.savez_compressed(path,**values)
    atomic_json(meta,dict(spec=spec,eye_mask_source=source,mask_sha256=sha(path),recovery_attempts=attempts))
    overlay=np.asarray(base).copy();eye=joint['eye_core']
    overlay[eye]=(overlay[eye]*.4+np.array([255,0,0])*.6).astype('uint8')
    Image.fromarray(overlay).save(MASK_ROOT/(rec['sha256']+'_eyes.png'))

def load_masks(rec):
    path,meta=mask_paths(rec);info=json.loads(need(meta).read_text())
    assert info['spec']==mask_spec(rec) and sha(need(path))==info['mask_sha256']
    with np.load(path,allow_pickle=False) as z:
        masks={k[3:]:z[k].copy() for k in z.files if k.startswith('m__')}
        joint={k[3:]:z[k].copy() for k in z.files if k.startswith('j__')}
    masks['raw_mask']=Image.fromarray(masks['raw_mask']);joint['raw_mask']=Image.fromarray(joint['raw_mask'])
    return masks,joint,info

def preflight_masks():
    global PENDING_MASKS
    PENDING_MASKS=[];seen={}
    for i,r in enumerate(BASE_RECORDS):
        digest=r['sha256']
        if digest in seen:
            if seen[digest]:PENDING_MASKS.append(dict(r,reason=seen[digest]))
            continue
        try:create_mask(r);seen[digest]=''
        except ValueError as error:
            if not str(error).startswith('Eye protection:'):raise
            seen[digest]=str(error);PENDING_MASKS.append(dict(r,reason=str(error),attempts=getattr(error,'eye_attempts',[])))
        if i%20==0:print('Mask check',i+1,'/',len(BASE_RECORDS),flush=True)
    atomic_json(OUT/'pending_eye_masks.json',PENDING_MASKS)
    print('Masks ready:',len(BASE_RECORDS)-len(PENDING_MASKS),'/',len(BASE_RECORDS),'| pending:',len(PENDING_MASKS))
    export_eye_debug()
    return PENDING_MASKS
preflight_masks()


## เฉพาะภาพที่ตรวจตาไม่ผ่าน: กำหนดกรอบตาและตรวจภาพ
ถ้าไม่มีภาพค้าง ระบบจะผ่านเซลล์นี้เอง ระบบจะทำภาพที่พร้อมในเซลล์ถัดไปก่อนได้ หากต้องการแก้ภาพค้าง ให้ปรับช่วง X/Y ของตาซ้าย–ขวาบนภาพ กดตรวจยืนยันแล้วบันทึกจนหมด จากนั้นรันเซลล์สร้างภาพด้านล่างต่อ การกำหนดมือถูกระบุแยกในรายงาน

In [ ]:
# Only images whose automatic eye detector failed need annotation.
# Coordinates determine masks; output eye pixels still come from that image itself.
if PENDING_MASKS:
    import ipywidgets as widgets
    from google.colab import output
    output.enable_custom_widget_manager()
    from IPython.display import display,clear_output
    unique={r['sha256']:r for r in PENDING_MASKS}
    choice=widgets.Dropdown(options=[(f'{r["split"]}/{r["trial"]}/{r["case"]}/{r["branch"]}',k) for k,r in unique.items()],description='ภาพ:')
    sliders=[widgets.IntRangeSlider(value=(0,0),min=0,max=1023,description=n,continuous_update=False,layout=widgets.Layout(width='95%')) for n in ['ตาซ้าย X','ตาซ้าย Y','ตาขวา X','ตาขวา Y']]
    checked=widgets.Checkbox(value=False,description='ตรวจภาพแล้ว: สี่เหลี่ยมครอบดวงตาทั้งสองถูกต้อง')
    save=widgets.Button(description='บันทึก mask ตา',button_style='success')
    preview=widgets.Output();status=widgets.Output()
    def selected_image():
        rec=unique[choice.value]
        with Image.open(rec['path']) as im:return rec,im.convert('RGB').copy()
    def update_preview(change=None):
        checked.value=False
        if choice.value is None:return
        rec,im=selected_image();draw=ImageDraw.Draw(im)
        for offset,color in [(0,'red'),(2,'cyan')]:
            x1,x2=sliders[offset].value;y1,y2=sliders[offset+1].value
            draw.rectangle((x1,y1,x2,y2),outline=color,width=4)
        with preview:
            clear_output(wait=True);display(ImageOps.contain(im,(720,720)))
    def change_image(change=None):
        if choice.value is None:return
        _,im=selected_image()
        for i,slider in enumerate(sliders):slider.value=(0,0);slider.max=(im.width if i%2==0 else im.height)-1
        update_preview()
    def save_annotation(button):
        with status:
            clear_output()
            if not checked.value:print('ปรับสี่เหลี่ยมทั้งสองให้ครอบตา แล้วติ๊กยืนยันตำแหน่งก่อนบันทึก');return
            rec,base=selected_image();points=np.zeros((68,2),np.float32)
            boxes=[]
            for offset,start in [(0,36),(2,42)]:
                x1,x2=sliders[offset].value;y1,y2=sliders[offset+1].value
                if x2-x1<4 or y2-y1<2:print('กรอบตาต้องมีความกว้างและสูง ไม่ใช่เส้นหรือจุด');return
                boxes.append((x1,y1,x2,y2))
                points[start:start+6]=[[x1,(y1+y2)/2],[x1,y1],[x2,y1],[x2,(y1+y2)/2],[x2,y2],[x1,y2]]
            if boxes[0][2]>=boxes[1][0]:print('กรอบตาซ้ายและขวาของภาพต้องแยกกัน ไม่ทับกัน');return
            build_eye_protection(base.size,points) # Same original mask constructor and margins.
            atomic_json(MANUAL_ROOT/(rec['sha256']+'.json'),dict(image_sha256=rec['sha256'],size=list(base.size),reviewed=True,landmarks=points.tolist(),annotation='manually reviewed eye rectangles; no image pixels replaced'))
            create_mask(rec)
            del unique[choice.value]
            choice.options=[(f'{r["split"]}/{r["trial"]}/{r["case"]}/{r["branch"]}',k) for k,r in unique.items()]
            if unique:print('บันทึกแล้ว เลือกภาพถัดไป เหลือ',len(unique),'ภาพไม่ซ้ำ')
            else:preflight_masks();print('ครบแล้ว: รันเซลล์ diffusion ด้านล่างต่อได้ ไม่ต้องรันเซลล์บนใหม่')
    for slider in sliders:slider.observe(update_preview,names='value')
    choice.observe(change_image,names='value');save.on_click(save_annotation)
    display(widgets.VBox([widgets.HTML('กรอบแดง = ตาด้านซ้ายของภาพ; กรอบฟ้า = ตาด้านขวาของภาพ ปรับให้ครอบเปลือกตา ดูภาพก่อนกดบันทึก'),choice,*sliders,preview,checked,save,status]))
    change_image()
else:print('Automatic eye masks complete. No manual annotation needed.')


## สร้างภาพด้วย original Joint diffusion
แสดงหนึ่งบรรทัดต่อภาพ โมเดลโหลดครั้งเดียว ไม่มีการเปลี่ยน seed จนกว่าจะได้ภาพที่ชอบ

In [ ]:
# Generate ready cases; keep unresolved cases explicit and block final comparisons until complete.
preflight_masks()
pending_hashes={r['sha256'] for r in PENDING_MASKS}
if pending_hashes:print('ทำภาพที่พร้อมก่อน; รายการรอตรวจอยู่ใน pending_eye_masks.json:',len(PENDING_MASKS))
from diffusers import StableDiffusionInpaintPipeline,UniPCMultistepScheduler
from IPython.utils.capture import capture_output
DIFFUSION_MODEL_ID='stablediffusionapi/realistic-vision-v51'
if 'inpaint_pipe' in globals():
    loaded=str(getattr(inpaint_pipe.config,'_name_or_path',''))
    if DIFFUSION_MODEL_ID.split('/')[-1] not in loaded:
        raise RuntimeError('Pipeline ที่โหลดอยู่ไม่ตรง Realistic Vision v5.1: เปิด runtime ใหม่เพื่อไม่ใช้โมเดลผิดรุ่น')
else:
    DIFFUSION_REVISION=HfApi().model_info(DIFFUSION_MODEL_ID).sha
    inpaint_pipe=StableDiffusionInpaintPipeline.from_pretrained(DIFFUSION_MODEL_ID,revision=DIFFUSION_REVISION,torch_dtype=torch.float16,safety_checker=None)
    inpaint_pipe.enable_model_cpu_offload()
# Exact original scheduler family + original checker setting. Memory offload only affects placement.
inpaint_pipe.scheduler=UniPCMultistepScheduler.from_config(inpaint_pipe.scheduler.config)
inpaint_pipe.safety_checker=None
if getattr(inpaint_pipe,'_execution_device',torch.device('cpu')).type=='cpu':inpaint_pipe.enable_model_cpu_offload()
if callable(getattr(inpaint_pipe.vae,'enable_slicing',None)):inpaint_pipe.vae.enable_slicing()
inpaint_pipe.set_progress_bar_config(disable=True)
validate_joint_prompt_lengths({PROMPT_TAG:PROMPT_CONFIG})
ENVIRONMENT=dict(torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),
    diffusion_model=str(getattr(inpaint_pipe.config,'_name_or_path','')),diffusion_revision=globals().get('DIFFUSION_REVISION'),
    segmentation_revision=getattr(seg_model.config,'_commit_hash',None),scheduler=type(inpaint_pipe.scheduler).__name__,safety_checker=None,
    memory_placement='CPU offload; original sampling/compositing functions unchanged')
(OUT/'environment.json').write_text(json.dumps(ENVIRONMENT,indent=2))
(OUT/'requirements.freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))

def key_of(r):return (r['split'],r['trial'],r['case'],r['branch'])
ledger=OUT/'outcomes.json';OUTCOMES=json.loads(ledger.read_text()) if ledger.exists() else []
done={key_of(r):r for r in OUTCOMES};assert len(done)==len(OUTCOMES)
for r in OUTCOMES:
    assert r['status']=='diffusion_success' and sha(need(Path(r['path'])))==r['sha256']
    base_rec=next(b for b in BASE_RECORDS if key_of(b)==key_of(r))
    _,_,info=load_masks(base_rec);assert info['mask_sha256']==r['mask_sha256'],'Mask changed since generation; use a new output run'
for rec in BASE_RECORDS:
    if key_of(rec) in done:continue
    if rec['sha256'] in pending_hashes:continue
    masks,joint,info=load_masks(rec)
    before_path=Path(rec['path']);assert sha(before_path)==rec['sha256']
    with Image.open(before_path) as im:base=im.convert('RGB').copy()
    started=time.perf_counter()
    try:
        # capture_output only hides repeated debug panels; original refine_joint/refine_once are untouched.
        with capture_output():
            after=refine_joint(base,masks,joint,PROMPT_CONFIG,SEED,MODE,**settings)
    except Exception as error:
        atomic_json(OUT/'last_generation_error.json',dict(split=rec['split'],trial=rec['trial'],case=rec['case'],branch=rec['branch'],error_type=type(error).__name__,message=str(error),no_fallback=True))
        raise
    original=np.asarray(base).astype(np.int16);result=np.asarray(after).astype(np.int16)
    if result.max()==0:raise RuntimeError('All-black output: stop instead of recording a fabricated success')
    alpha=joint_alpha(joint,MODE);eye=joint['eye_core'];hair=masks['hair']
    assert np.array_equal(original[eye],result[eye]),'Original eye-protection invariant failed'
    assert np.array_equal(original[alpha==0],result[alpha==0]),'Original outside-edit invariant failed'
    folder=before_path.parent;after_path=folder/'after.png';after.save(after_path)
    Image.fromarray((eye*255).astype('uint8')).save(folder/'eye_core.png')
    Image.fromarray(np.rint(alpha*255).astype('uint8')).save(folder/'edit_alpha.png')
    overlay=np.asarray(base).copy();overlay[eye]=(overlay[eye]*.4+np.array([255,0,0])*.6).astype('uint8')
    Image.fromarray(overlay).save(folder/'eye_overlay.png')
    joint['raw_mask'].save(folder/'diffusion_mask.png')
    row=dict(**{k:rec[k] for k in ['split','trial','case','branch','epoch']},path=str(after_path),before_path=str(before_path),sha256=sha(after_path),
        status='diffusion_success',reason='',seconds=time.perf_counter()-started,prompt=PROMPT_TAG,seed=SEED,
        mask_sha256=info['mask_sha256'],eye_mask_source=info['eye_mask_source'],eye_max_abs_RGB_change=int(np.abs(result-original)[eye].max()),
        hair_RGB_MAE_vs_before=float(np.abs(result-original)[hair].mean()/255.) if hair.any() else None,
        outside_edit_max_abs_RGB_change=int(np.abs(result-original)[alpha==0].max()) if (alpha==0).any() else None)
    OUTCOMES.append(row);done[key_of(row)]=row;atomic_json(ledger,OUTCOMES)
    print(f'{len(done)}/{len(BASE_RECORDS)}',rec['split'],rec['trial'],rec['case'],rec['branch'],'original Joint done',flush=True)
if PENDING_MASKS:
    pd.DataFrame(OUTCOMES).to_csv(OUT/'partial_outcomes.csv',index=False)
    debug_archive=export_eye_debug()
    try:files.download(debug_archive)
    except Exception as download_error:print('ดาวน์โหลดอัตโนมัติไม่ได้ ให้เปิดไฟล์จาก Drive:',debug_archive)
    raise RuntimeError(f'บันทึกผลสำเร็จแล้ว {len(done)}/{len(BASE_RECORDS)}; ยังมี {len(PENDING_MASKS)} รายการรอตรวจตา ให้ใช้ช่องตรวจภาพแล้วรันเซลล์นี้ต่อ ผลที่เสร็จแล้วไม่ทำซ้ำ และยังไม่สรุปผลเปรียบเทียบจากข้อมูลไม่ครบ')
assert set(done)=={key_of(r) for r in BASE_RECORDS} and len(OUTCOMES)==len(BASE_RECORDS)
pd.DataFrame(OUTCOMES).to_csv(OUT/'all_outcomes.csv',index=False)
pd.DataFrame(OUTCOMES).groupby(['split','trial','branch','eye_mask_source']).size().unstack(fill_value=0).to_csv(OUT/'mask_source_counts.csv')
if hasattr(inpaint_pipe,'remove_all_hooks'):inpaint_pipe.remove_all_hooks()
inpaint_pipe.to('cpu');seg_model.cpu();gc.collect();torch.cuda.empty_cache()
id_net.cuda().eval().requires_grad_(False);parser_net.cuda().eval().requires_grad_(False)


In [ ]:
@torch.no_grad()
def image_tensor(path):
    with Image.open(path) as im:a=np.asarray(im.convert('RGB')).copy()
    return torch.from_numpy(a).float().permute(2,0,1)[None].cuda()/255.

REFERENCE_CACHE={};SOURCE_EYE_FAILURES=[]
def shared_reference(split,key):
    cache_key=(split,key)
    if cache_key not in REFERENCE_CACHE:
        folder=REPORT_ROOT/split/'images'/key
        d={name:image_tensor(folder/(name+'.png')).cpu() for name in ['face','ref']}
        for name in ['target','protected','eval_core','eval_refmask']:d[name]=read_mask(folder/(name+'.png'))
        assert d['face'].shape[-2:]==(256,256) and d['ref'].shape[-2:]==(256,256)
        with Image.open(folder/'face.png') as im:source=im.convert('RGB').copy()
        try:
            eye=build_eye_protection(source.size,detect_eye_landmarks(source))['eye_core']
        except ValueError as error:
            if not str(error).startswith('Eye protection:'):raise
            eye=np.zeros((source.height,source.width),dtype=bool)
            SOURCE_EYE_FAILURES.append(dict(split=split,case=key,status='source_eye_metric_unavailable',message=str(error)))
            pd.DataFrame(SOURCE_EYE_FAILURES).to_csv(Path(OUT)/'source_eye_metric_failures.csv',index=False)
        d['source_eye']=torch.from_numpy(eye.copy())[None,None]
        for name in ['eval_core','eval_refmask','protected']:assert d[name].any()
        REFERENCE_CACHE[cache_key]=d
    return gpu(REFERENCE_CACHE[cache_key])

@torch.no_grad()
def predicted_hair(im):
    labels=segment(align.to_bisenet(im))
    return F.interpolate((labels==13).float().reshape(1,1,*labels.shape[-2:]),(256,256),mode='nearest')>.5

@torch.no_grad()
def metrics_for(path,rec):
    full=image_tensor(path);im=to_size(full,256)
    d=shared_reference(rec['split'],rec['case'])
    L,ab=quantile_color(im,d['ref'],d['eval_core'],d['eval_refmask'])
    vals=dict(L_quantile=float(L),ab_quantile=float(ab),
        identity_cosine=float((id_features(im)*id_features(d['face'])).sum(1).mean()),
        protected_RGB_MAE_vs_input=float(masked_mae(im,d['face'],d['protected'])),
        source_eye_RGB_MAE=float(masked_mae(im,d['face'],d['source_eye'])) if d['source_eye'].any() else float('nan'))
    vals.update(extra_metrics(im,d))
    vals.update({'proxy_'+k:v for k,v in shape_metrics(im,d).items()})
    pred=predicted_hair(im).cpu()
    assert all(np.isfinite(v) for k,v in vals.items() if k!='source_eye_RGB_MAE')
    return vals,pred



## เมทริกและกราฟก่อน–หลัง บนคู่ข้อมูลเดียวกันครบทุกแขน

In [ ]:
METRIC_ROWS=[]
for rec in BASE_RECORDS:
    for stage,path in [('before',rec['path']),('after',done[key_of(rec)]['path'])]:
        vals,_=metrics_for(path,rec)
        METRIC_ROWS.append(dict(**{k:rec[k] for k in ['split','trial','case','branch','epoch']},stage=stage,
            outcome='input' if stage=='before' else done[key_of(rec)]['status'],**vals))
        pd.DataFrame(METRIC_ROWS).to_csv(OUT/'metrics.partial.csv',index=False)
frame=pd.DataFrame(METRIC_ROWS)
assert len(frame)==2*len(BASE_RECORDS)
assert not frame.duplicated(['split','trial','case','branch','stage']).any()
frame.to_csv(OUT/'per_image_metrics.csv',index=False);(OUT/'metrics.partial.csv').unlink()
MAIN=[('L_quantile','Lightness error L ↓'),('ab_quantile','Color error ab ↓'),('identity_cosine','Face identity cosine ↑'),
 ('protected_RGB_MAE_vs_input','Protected-region MAE ↓'),('protected_SSIM','Protected-region SSIM ↑'),('protected_PSNR','Protected-region PSNR ↑')]
PROXY=[('proxy_target_mask_IoU','HFG-derived proxy IoU ↑'),('proxy_target_Dice','HFG-derived proxy Dice ↑'),('proxy_boundary_F1_2px','HFG-derived proxy boundary F1 ↑')]
metrics=[k for k,_ in MAIN+PROXY]+['source_eye_RGB_MAE']
means=frame.groupby(['split','trial','branch','stage'])[metrics].mean()
sds=frame.groupby(['split','trial','branch','stage'])[metrics].std(ddof=1)
means.to_csv(OUT/'mean_metrics_all_cases.csv');sds.to_csv(OUT/'sd_across_cases.csv')
frame.groupby(['split','trial','branch','stage'])[metrics].count().to_csv(OUT/'valid_case_count_per_metric.csv')
deltas=[]
for rec in BASE_RECORDS:
    subset=frame[(frame.split==rec['split'])&(frame.trial==rec['trial'])&(frame.case==rec['case'])&(frame.branch==rec['branch'])].set_index('stage')
    assert set(subset.index)=={'before','after'}
    deltas.append(dict(**{k:rec[k] for k in ['split','trial','case','branch']},status=done[key_of(rec)]['status'],
        **{m:float(subset.loc['after',m]-subset.loc['before',m]) for m in metrics}))
pd.DataFrame(deltas).to_csv(OUT/'paired_after_minus_before.csv',index=False)
for split,keys in SPLITS.items():
    for trial in TRIALS:
        for branch in BRANCHES:
            for stage in ['before','after']:
                part=frame[(frame.split==split)&(frame.trial==trial)&(frame.branch==branch)&(frame.stage==stage)]
                assert set(part.case)==set(keys) and len(part)==len(keys),'Missing case before final report'
rotation_parts=[]
for stage in ['before','after']:
    part=frame[frame.stage==stage]
    keys=['split','trial','case','epoch']
    on=part[part.branch=='HFG_rotate_on'][keys+metrics]
    off=part[part.branch=='HFG_rotate_off'][keys+metrics]
    paired=off.merge(on,on=keys,how='outer',suffixes=('_off','_on'),indicator=True,validate='one_to_one')
    if not paired['_merge'].eq('both').all():
        paired.to_csv(OUT/'rotation_pairing_error.csv',index=False)
        raise RuntimeError('Incomplete rotation pairs: inspect rotation_pairing_error.csv; no imputation')
    delta=paired[keys].copy();delta['stage']=stage
    for metric in metrics:delta[metric]=paired[metric+'_off']-paired[metric+'_on']
    rotation_parts.append(delta)
pd.concat(rotation_parts,ignore_index=True).to_csv(OUT/'paired_rotation_off_minus_on.csv',index=False)

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display
COLORS=['#2875BA','#7E57C2','#D77A29']
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,'figure.facecolor':'white'})
with PdfPages(OUT/'all_cases_report.pdf') as pdf:
    for split,keys in SPLITS.items():
        for trial in TRIALS:
            folder=OUT/'figures'/split/trial;folder.mkdir(parents=True,exist_ok=True)
            count_info=[]
            for branch in BRANCHES:
                records=[r for r in OUTCOMES if r['split']==split and r['trial']==trial and r['branch']==branch]
                count_info.append(f'{branch}: diffusion {sum(r["status"]=="diffusion_success" for r in records)}/{len(keys)}, manual eye masks {sum(r['eye_mask_source']=='manual_reviewed' for r in records)}')
            for group,specs in [('primary',MAIN),('shape_proxy',PROXY)]:
                fig,axes=plt.subplots(len(specs)//3,3,figsize=(16,4.8*(len(specs)//3)),squeeze=False)
                for ax,(metric,title) in zip(axes.flat,specs):
                    for j,branch in enumerate(BRANCHES):
                        ks=[(split,trial,branch,stage) for stage in ['before','after']]
                        ax.errorbar([j*3,j*3+1],[means.loc[k,metric] for k in ks],yerr=[sds.loc[k,metric] for k in ks],fmt='o-',color=COLORS[j],capsize=3)
                    ax.set_xticks([0,1,3,4,6,7],['ON\nbefore','ON\nafter','OFF\nbefore','OFF\nafter','BS\nbefore','BS\nafter'])
                    ax.set_title(title,loc='left');ax.grid(alpha=.2);ax.spines[['top','right']].set_visible(False)
                fig.suptitle(f'{split} | {trial} | ALL {len(keys)} cases, mean ± SD | Original Joint method; no fallback images')
                fig.text(.03,.015,' | '.join(count_info),fontsize=8)
                fig.tight_layout(rect=(0,.055,1,.94));fig.savefig(folder/(group+'.png'),dpi=170);pdf.savefig(fig);plt.close(fig)
            for key in keys:
                fig,axes=plt.subplots(3,2,figsize=(10,13))
                for i,branch in enumerate(BRANCHES):
                    rec=done[(split,trial,key,branch)]
                    for j,(path,label) in enumerate([(rec['before_path'],'Before'),(rec['path'],rec['status'])]):
                        with Image.open(path) as im:axes[i,j].imshow(im)
                        axes[i,j].set_title(branch+' | '+label,fontsize=10);axes[i,j].axis('off')
                fig.suptitle(f'{split}/{key} | {trial}');fig.tight_layout()
                fig.savefig(folder/(key+'.png'),dpi=130);pdf.savefig(fig);plt.close(fig)
for path,digest in INPUT_HASHES.items():assert sha(Path(path))==digest
manifest=dict(status='complete',scheduled_cases={s:len(k) for s,k in SPLITS.items()},expected_after=len(BASE_RECORDS),actual_after=len(OUTCOMES),
    actual_metric_rows=len(frame),diffusion_success=sum(r['status']=='diffusion_success' for r in OUTCOMES),
    manual_eye_masks=sum(r['eye_mask_source']=='manual_reviewed' for r in OUTCOMES),fallback_original=0,original_helper_sha256=ORIGINAL_HELPER_SHA256,config=CONFIG,
    no_training=True,no_cases_excluded=True,source_images_unchanged=True,checkpoint_lock_from_source=SOURCE_LOCK,
    evaluation='original Joint diffusion on every scheduled input; no substitution; validated eye/outside-edit invariants',
    limitations=['hair RGB is editable; color retention is a prompt request, measured rather than guaranteed','shape target is HFG-derived proxy',
                 'no external sharpening; original joint mask controls editable regions','previously viewed test; no tuning or selection from test',
                 'source-eye metric can be unavailable independently of other metrics'])
(OUT/'completion.json').write_text(json.dumps(manifest,indent=2))
(OUT/'README_TH.txt').write_text("""Original EyeModes Joint batch
Copied original helper cell without altering prompts, masks, diffusion or compositing.
One prompt6_combined / seed3407. No hair RGB lock, color correction, external sharpening or fallback.
HairFastGAN ON/OFF and Barbershop pre-diffusion images reused; checkpoint epochs are inherited from the source report.
Failed original eye detections use fixed resize/pad retries with the same detector and inverse coordinate mapping. Recovery source and attempts are recorded per image.
Manual eye annotations, when needed, are reported separately; eye content always comes from the branch's own pre-diffusion image.
All expected outputs must exist before summary. Test is previously viewed; no selection from test.
Mask-based shape scores are HFG-derived proxies, not independent ground truth.
Read paired_after_minus_before.csv and paired_rotation_off_minus_on.csv alongside images.
""",encoding='utf-8')
archive=shutil.make_archive(str(OUT),'zip',root_dir=OUT)
display(means);display(pd.DataFrame(OUTCOMES).groupby(['split','trial','branch','status']).size())
print('COMPLETE:',manifest,'\nZIP:',archive)
files.download(archive)
